# VLM Risk Pipeline Ibrida per Video Industriali

Questo notebook implementa una pipeline sperimentale **ibrida** per la valutazione del rischio in video industriali tramite Google Gemini, progettata come artefatto di ricerca per tesi. La pipeline unifica una **parte comune VLM** e due **rami quasi paralleli**: un ramo in cui il VLM assegna lo score istantaneo e un ramo in cui lo score istantaneo è calcolato dal codice tramite **AHP + funzioni monotone** [file:299][file:300].

## Architettura logica

### Parte comune iniziale
1. Ingestione video
2. Windowing temporale

### Prima biforcazione dopo il windowing
- **Ramo VLM libero**: parte direttamente dalla clip/finestra temporale con prompt descrittivo-causale libero, poi scoring istantaneo VLM, smoothing nel codice e semaforo nel codice.
- **Ramo guidato comune a VLM guidato e AHP**:
  3. Prompt descrittivo (VLM)
  4. Prompt question set diagnostico aggiornato (VLM)
  5. Motore causale leggero (VLM)

### Seconda biforcazione dei rami guidati
- **Ramo VLM guidato**: scoring istantaneo assegnato dal modello, smoothing nel codice, semaforo nel codice [file:299]
- **Ramo VLM guidato + AHP**: scoring istantaneo calcolato nel codice da risk cues, smoothing nel codice, semaforo nel codice [file:300]

L'obiettivo metodologico è confrontare tre rami attivabili sulla stessa segmentazione temporale: un ramo libero metodologicamente autonomo dalla sola clip e due rami guidati che condividono la stessa base informativa descrittivo-diagnostica-causale [file:299][file:300].

## 1. Installazione dipendenze

Questa cella installa i pacchetti necessari per:
- accesso a Google Gemini;
- gestione video con OpenCV;
- export Excel con dashboard e grafici;
- gestione notebook e progress bar.

In [ ]:
!pip install -q google-genai opencv-python openpyxl tqdm python-dotenv numpy pandas matplotlib

## 2. Import e inizializzazione client Gemini

La chiave API viene letta dal file `.env` oppure dalle variabili d'ambiente. Il notebook usa la struttura robusta del notebook AHP come base, includendo retry, backoff, polling dei file caricati e parsing JSON difensivo [file:300].

In [ ]:
import os
import re
import json
import time
import math
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GEMINIAPIKEY") or os.getenv("GEMINI_APIKEY")

if not API_KEY:
    raise ValueError(
        "Chiave API non trovata. Crea un file .env con una tra le seguenti righe:\n"
        "GEMINI_API_KEY=la_tua_chiave oppure GEMINIAPIKEY=la_tua_chiave"
    )

client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash"
print(f"Client Gemini inizializzato. Modello: {MODEL_NAME}")

## 3. Configurazione parametri sperimentali

Questa sezione raccoglie tutti i parametri modificabili dell'esperimento. I valori iniziali mantengono la logica dei notebook precedenti: finestra di 4 secondi e stride di 2 secondi, quindi **2 secondi di sovrapposizione** tra finestre consecutive [file:299][file:300].

Il parametro `OUTPUT_LANGUAGE` controlla la lingua dei contenuti testuali restituiti dal modello, senza alterare la logica interna della pipeline.

In [ ]:

# =========================
# CONFIGURAZIONE ESPERIMENTO
# =========================

VIDEO_PATH = r"- - -"  # Modificare con il percorso reale

# Windowing temporale
WINDOW_SECONDS = 4
STRIDE_SECONDS = 2
MAX_WINDOWS = None

# Parametri Gemini
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 3500

# Limiti lunghezza prompt
MAX_CHARS_DIAGNOSTIC = 1200
MAX_CHARS_QUESTION_SET = 1800
MAX_CHARS_CAUSAL = 1200
MAX_CHARS_DECISION_VLM = 1200
MAX_CHARS_FREE_DESCRIPTIVE = 2200
MAX_CHARS_FREE_SCORING = 1200

# Lingua output
OUTPUT_LANGUAGE = "italiano"  # Esempi: "italiano", "inglese"

# Abilitazione rami per debug/test rapidi
ENABLE_VLM_FREE_BRANCH = True
ENABLE_VLM_BRANCH = True
ENABLE_AHP_BRANCH = False

# Smoothing VLM guidato
ALPHA_VLM = 0.4
BETA_VLM = 0.25

# Smoothing AHP
ALPHA_AHP = 0.4
BETA_AHP = 0.25

# Smoothing VLM libero: di default replica il comportamento del ramo VLM
ALPHA_VLM_FREE = 0.4
BETA_VLM_FREE = 0.25

# Uso opzionale cue confidence nello scoring AHP
USE_CUE_CONFIDENCE = False

# Parametri funzioni monotone AHP per S1-S7

# S1 -> Lineare
# Nessun parametro necessario

# S2 -> Crescente a tassi decrescenti
GAMMA_S2 = 3.0

# S3 -> Sigmoide
K_S3 = 10.0
TAU_S3 = 0.35

# S4 -> Sigmoide
K_S4 = 12.0
TAU_S4 = 0.40

# S5 -> Crescente a tassi crescenti
GAMMA_S5 = 2.4

# S6 -> Sigmoide
K_S6 = 12.0
TAU_S6 = 0.30

# S7 -> Crescente a tassi crescenti
GAMMA_S7 = 2.2

# Soglie semaforo applicate allo score dinamico
THRESH_GREEN = (0, 24)
THRESH_YELLOW = (25, 49)
THRESH_ORANGE = (50, 74)
THRESH_RED = (75, 100)

AHP_WEIGHTS = {
    "S1": 0.1109,
    "S2": 0.0370,
    "S3": 0.0549,
    "S4": 0.1538,
    "S5": 0.2228,
    "S6": 0.3447,
    "S7": 0.0759,
}

RISK_LABELS = {
    "S1": "Interferenza tra persone e mezzi di movimentazione negli spazi comuni",
    "S2": "Riduzione della visibilità durante lo svolgimento di un'attività",
    "S3": "Comportamenti non conformi durante lo svolgimento di un'attività",
    "S4": "Interferenza tra mezzo e infrastrutture interne all'impianto",
    "S5": "Instabilità di elementi strutturali o di stoccaggio soggetti a urto o sovraccarico",
    "S6": "Accesso a organi meccanici in movimento durante operazioni manuali",
    "S7": "Movimentazione manuale di carichi pesanti o instabili senza ausili meccanici adeguati",
}

RISK_CUE_KEYS = {
    "S1": "S1_people_vehicle_interference",
    "S2": "S2_visibility_attention_reduction",
    "S3": "S3_non_compliant_behavior",
    "S4": "S4_vehicle_infrastructure_interference",
    "S5": "S5_storage_structural_instability",
    "S6": "S6_access_to_moving_mechanical_parts",
    "S7": "S7_manual_handling_heavy_unstable_loads",
}

WORKDIR = Path("pipeline_workdir_hybrid")
CLIPS_DIR = WORKDIR / "clips"
OUT_DIR = WORKDIR / "results"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

video_path = Path(VIDEO_PATH)
print(f"VIDEO_PATH: {video_path}")
print(f"WINDOW_SECONDS={WINDOW_SECONDS}, STRIDE_SECONDS={STRIDE_SECONDS}, overlap={WINDOW_SECONDS-STRIDE_SECONDS}")
print(f"OUTPUT_LANGUAGE={OUTPUT_LANGUAGE}")
print(
    f"ENABLE_VLM_BRANCH={ENABLE_VLM_BRANCH}, "
    f"ENABLE_AHP_BRANCH={ENABLE_AHP_BRANCH}, "
    f"ENABLE_VLM_FREE_BRANCH={ENABLE_VLM_FREE_BRANCH}"
)


## 4. Ingestione video

Il video viene aperto una sola volta per leggere fps, durata, risoluzione e numero di frame. Le clip saranno poi estratte finestra per finestra durante l'esecuzione della pipeline.

In [ ]:
def ingest_video(path: Path) -> dict:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Impossibile aprire il video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = round(frame_count / fps, 3) if fps else 0.0
    cap.release()
    return {
        "fps": fps,
        "frames": frame_count,
        "width": width,
        "height": height,
        "duration": duration,
    }

if video_path.exists():
    video_meta = ingest_video(video_path)
    print("Metadati video:")
    for k, v in video_meta.items():
        print(f"- {k:>8}: {v}")
else:
    print("ATTENZIONE: video non trovato. Aggiorna VIDEO_PATH prima di eseguire la pipeline completa.")
    video_meta = {"fps": 25.0, "frames": 0, "width": 0, "height": 0, "duration": 0.0}

## 5. Suddivisione in finestre temporali sovrapposte

Con `WINDOW_SECONDS = 4` e `STRIDE_SECONDS = 2`, ogni nuova finestra si sovrappone alla precedente di 2 secondi. Questa scelta riduce il rischio di perdere eventi rilevanti che si sviluppano a cavallo tra due segmenti adiacenti [file:299][file:300].

In [ ]:
def build_windows(duration: float, win: float, stride: float) -> list:
    windows = []
    start = 0.0
    idx = 0
    while start < duration:
        end = min(start + win, duration)
        if end - start >= 0.5:
            windows.append({
                "id": idx,
                "start": round(start, 3),
                "end": round(end, 3),
                "duration": round(end - start, 3),
            })
            idx += 1
        start += stride
    return windows


def extract_clip(src: Path, dst: Path, start: float, end: float) -> None:
    cap = cv2.VideoCapture(str(src))
    if not cap.isOpened():
        raise RuntimeError(f"Impossibile aprire il video sorgente: {src}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        str(dst),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    start_frame = int(start * fps)
    end_frame = int(end * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    cur = start_frame

    while cur < end_frame:
        ok, frame = cap.read()
        if not ok:
            break
        out.write(frame)
        cur += 1

    out.release()
    cap.release()

windows = build_windows(video_meta["duration"], WINDOW_SECONDS, STRIDE_SECONDS)
if MAX_WINDOWS is not None:
    windows = windows[:MAX_WINDOWS]

print(f"Finestre generate: {len(windows)}")
pd.DataFrame(windows).head(10)

## 6. Utility per chiamate Gemini

Questa sezione riusa l'impostazione più robusta del notebook AHP: parser JSON difensivo, polling dello stato dei file caricati, retry automatico con backoff esponenziale e gestione degli errori transitori [file:300].

In [ ]:
# =========================
# 6. Utility robuste Gemini + parsing JSON difensivo
# =========================

FILE_POLL_EVERY_SECONDS = 2
FILE_ACTIVE_TIMEOUT_SECONDS = 300
MAX_RETRIES_VIDEO = 99
MAX_RETRIES_TEXT = 99
INITIAL_BACKOFF_SECONDS = 1.5
MAX_BACKOFF_SECONDS = 30.0
REQUEST_TIMEOUT_MS = 180000
FALLBACK_MODEL_NAME = None

try:
    HTTP_OPTIONS = types.HttpOptions(timeout=REQUEST_TIMEOUT_MS)
except Exception:
    HTTP_OPTIONS = None

try:
    GEN_CONFIG = types.GenerateContentConfig(
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        response_mime_type="application/json",
        http_options=HTTP_OPTIONS,
    )
except Exception:
    GEN_CONFIG = types.GenerateContentConfig(
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        response_mime_type="application/json",
    )

TOKEN_AUDIT_LOGS: List[Dict[str, Any]] = []
_BRANCH_API_CALL_COUNTERS: Dict[str, int] = {
    "common": 0,
    "vlm_guided": 0,
    "ahp": 0,
    "vlm_free": 0,
}
_GLOBAL_EXECUTION_ORDER = 0


def extract_json_object(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None

    raw = str(text).strip()
    if not raw:
        return None

    raw = re.sub(r"```json", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"```", "", raw).strip()

    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end == -1 or end <= start:
        return None

    return raw[start:end + 1].strip()


def try_parse_json_strict(text: Optional[str]) -> Optional[dict]:
    if text is None:
        return None

    raw = str(text).strip()
    if not raw:
        return None

    raw = re.sub(r"```json", "", raw, flags=re.IGNORECASE)
    raw = re.sub(r"```", "", raw).strip()

    candidate = extract_json_object(raw) or raw

    try:
        parsed = json.loads(candidate)
        return parsed if isinstance(parsed, dict) else {"data": parsed}
    except Exception as e:
        print(f"JSON trovato ma non valido: {e}")
        return None


def safe_json(text: Optional[str]) -> dict:
    """
    Mantiene compatibilità con il resto del notebook:
    - restituisce dict valido se possibile;
    - altrimenti {}.
    Nessun tentativo di repair: il parsing è stretto.
    """
    parsed = try_parse_json_strict(text)

    if parsed is not None:
        return parsed

    raw = "" if text is None else str(text).strip()
    if text is None:
        print("Risposta vuota dal modello: None")
    elif not raw:
        print("Risposta vuota dal modello: stringa vuota")
    else:
        print("Parsing JSON fallito. Possibile output troncato o malformato.")
        print("Inizio risposta:", raw[:300])
        print("Fine risposta:", raw[-300:])

    return {}


def safe_json_or_none(text: Optional[str], debug: bool = True) -> Optional[dict]:
    """
    Versione stretta: ritorna None se il JSON non è valido.
    Serve per i casi in cui vogliamo rigenerare finché il JSON non è coerente.
    Nessun tentativo di repair.
    """
    parsed = try_parse_json_strict(text)

    if parsed is not None:
        return parsed

    if debug:
        raw = "" if text is None else str(text).strip()
        if text is None:
            print("Risposta vuota dal modello: None")
        elif not raw:
            print("Risposta vuota dal modello: stringa vuota")
        else:
            print("Parsing JSON fallito. Possibile output troncato o malformato.")
            print("Inizio risposta:", raw[:500])
            print("Fine risposta:", raw[-500:])

    return None


def compute_backoff(
    attempt: int,
    initial_delay: float = INITIAL_BACKOFF_SECONDS,
    max_delay: float = MAX_BACKOFF_SECONDS,
) -> float:
    base = min(max_delay, initial_delay * (2 ** attempt))
    jitter = random.uniform(0, 0.35 * base)
    return min(max_delay, base + jitter)


def is_retryable_error(err: Exception) -> bool:
    msg = str(err).lower()
    retryable_markers = [
        "408", "429", "500", "502", "503", "504",
        "unavailable", "resourceexhausted", "timeout", "deadlineexceeded",
        "temporarily unavailable", "high demand", "connection reset",
        "remoteprotocolerror", "ssl", "eof", "winerror 10054",
        "broken pipe", "service is currently unavailable",
    ]
    return any(marker in msg for marker in retryable_markers)


def rebuild_client():
    global client
    client = genai.Client(api_key=API_KEY)
    return client


def wait_until_file_active(
    file_obj,
    poll_every: int = FILE_POLL_EVERY_SECONDS,
    timeout: int = FILE_ACTIVE_TIMEOUT_SECONDS,
):
    start_time = time.time()

    while True:
        current = client.files.get(name=file_obj.name)
        state = getattr(current.state, "name", str(current.state))

        if state == "ACTIVE":
            return current

        if state == "FAILED":
            raise RuntimeError(f"File Gemini in stato FAILED: {file_obj.name}")

        if time.time() - start_time > timeout:
            raise TimeoutError(
                f"Timeout attesa attivazione file Gemini: {file_obj.name} stato={state}"
            )

        print(f"File in stato {state}, attendo {poll_every}s...")
        time.sleep(poll_every)


def upload_clip(clip_path: Path, max_retries: int = 5):
    last_err = None

    for attempt in range(max_retries):
        try:
            file_obj = client.files.upload(file=str(clip_path))
            return wait_until_file_active(file_obj)

        except Exception as e:
            last_err = e

            if is_retryable_error(e):
                try:
                    rebuild_client()
                except Exception:
                    pass

            if (not is_retryable_error(e)) or attempt == max_retries - 1:
                raise

            wait_s = compute_backoff(attempt)
            print(f"Retry upload clip {attempt + 1}/{max_retries} dopo errore: {e}")
            print(f"Attendo {wait_s:.1f}s...")
            time.sleep(wait_s)

    raise last_err


def delete_remote_clip(file_obj):
    try:
        client.files.delete(name=file_obj.name)
    except Exception as e:
        print(f"Impossibile eliminare file remoto: {e}")


def generate_content_with_model(model_name: str, contents, config_obj):
    return client.models.generate_content(
        model=model_name,
        contents=contents,
        config=config_obj,
    )


def estimate_token_count(text: Optional[str]) -> int:
    if text is None:
        return 0
    s = str(text).strip()
    if not s:
        return 0
    return max(1, int(round(len(s) / 4)))


def extract_usage_metadata(response) -> Dict[str, Optional[int]]:
    usage = getattr(response, "usage_metadata", None)
    if usage is None:
        return {
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "usage_source": "fallback_estimate",
        }

    candidates_in = [
        "prompt_token_count",
        "input_token_count",
        "prompt_tokens",
        "input_tokens",
        "cached_content_token_count",
    ]
    candidates_out = [
        "candidates_token_count",
        "output_token_count",
        "completion_tokens",
        "output_tokens",
    ]
    candidates_total = [
        "total_token_count",
        "total_tokens",
    ]

    def pick(obj, names):
        for name in names:
            value = getattr(obj, name, None)
            if value is not None:
                try:
                    return int(value)
                except Exception:
                    pass
        return None

    input_tokens = pick(usage, candidates_in)
    output_tokens = pick(usage, candidates_out)
    total_tokens = pick(usage, candidates_total)

    if total_tokens is None and input_tokens is not None and output_tokens is not None:
        total_tokens = int(input_tokens + output_tokens)

    usage_source = (
        "sdk_usage_metadata"
        if any(v is not None for v in [input_tokens, output_tokens, total_tokens])
        else "fallback_estimate"
    )

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "usage_source": usage_source,
    }


def register_token_usage(
    branch_name: str,
    block_name: str,
    window_id: int,
    model_name: str,
    prompt_text: Optional[str],
    response_text: Optional[str],
    response,
    status: str,
    error_message: str = "",
) -> Dict[str, Any]:
    global _GLOBAL_EXECUTION_ORDER

    usage = extract_usage_metadata(response)
    input_tokens = usage["input_tokens"]
    output_tokens = usage["output_tokens"]
    total_tokens = usage["total_tokens"]

    if input_tokens is None:
        input_tokens = estimate_token_count(prompt_text)
    if output_tokens is None:
        output_tokens = estimate_token_count(response_text)
    if total_tokens is None:
        total_tokens = int(input_tokens + output_tokens)

    _GLOBAL_EXECUTION_ORDER += 1
    api_call_number = _BRANCH_API_CALL_COUNTERS.get(branch_name, 0)

    entry = {
        "execution_order": _GLOBAL_EXECUTION_ORDER,
        "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "branch_name": branch_name,
        "block_name": block_name,
        "window_id": window_id,
        "model_name": model_name,
        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "total_tokens": int(total_tokens),
        "usage_source": usage.get("usage_source", "fallback_estimate"),
        "status": status,
        "error": error_message,
        "branch_api_call_count": api_call_number,
    }

    TOKEN_AUDIT_LOGS.append(entry)
    return entry


def call_gemini_video_model_raw(
    file_obj,
    prompt: str,
    model_name: str,
    config_obj,
    max_retries: int = MAX_RETRIES_VIDEO,
    branch_name: str = "common",
    block_name: str = "video_block",
    window_id: Optional[int] = None,
) -> Tuple[Any, str]:
    last_err = None

    for attempt in range(max_retries):
        try:
            _BRANCH_API_CALL_COUNTERS[branch_name] = (
                _BRANCH_API_CALL_COUNTERS.get(branch_name, 0) + 1
            )

            response = generate_content_with_model(
                model_name=model_name,
                contents=[file_obj, prompt],
                config_obj=config_obj,
            )
            text = getattr(response, "text", None)

            register_token_usage(
                branch_name=branch_name,
                block_name=block_name,
                window_id=-1 if window_id is None else int(window_id),
                model_name=model_name,
                prompt_text=prompt,
                response_text=text,
                response=response,
                status="success",
                error_message="",
            )

            return response, "" if text is None else str(text)

        except KeyboardInterrupt:
            raise

        except Exception as e:
            last_err = e

            if attempt == max_retries - 1 or not is_retryable_error(e):
                register_token_usage(
                    branch_name=branch_name,
                    block_name=block_name,
                    window_id=-1 if window_id is None else int(window_id),
                    model_name=model_name,
                    prompt_text=prompt,
                    response_text="",
                    response=None,
                    status="error",
                    error_message=str(e),
                )
                raise

            try:
                rebuild_client()
            except Exception:
                pass

            wait_s = compute_backoff(attempt)
            print(f"Retry video {attempt + 1}/{max_retries} dopo errore: {e}")
            print(f"Attendo {wait_s:.1f}s...")
            time.sleep(wait_s)

    raise last_err


def call_gemini_text_model_raw(
    prompt: str,
    model_name: str,
    config_obj,
    max_retries: int = MAX_RETRIES_TEXT,
    branch_name: str = "common",
    block_name: str = "text_block",
    window_id: Optional[int] = None,
) -> Tuple[Any, str]:
    last_err = None

    for attempt in range(max_retries):
        try:
            _BRANCH_API_CALL_COUNTERS[branch_name] = (
                _BRANCH_API_CALL_COUNTERS.get(branch_name, 0) + 1
            )

            response = generate_content_with_model(
                model_name=model_name,
                contents=prompt,
                config_obj=config_obj,
            )
            text = getattr(response, "text", None)

            register_token_usage(
                branch_name=branch_name,
                block_name=block_name,
                window_id=-1 if window_id is None else int(window_id),
                model_name=model_name,
                prompt_text=prompt,
                response_text=text,
                response=response,
                status="success",
                error_message="",
            )

            return response, "" if text is None else str(text)

        except KeyboardInterrupt:
            raise

        except Exception as e:
            last_err = e

            if attempt == max_retries - 1 or not is_retryable_error(e):
                register_token_usage(
                    branch_name=branch_name,
                    block_name=block_name,
                    window_id=-1 if window_id is None else int(window_id),
                    model_name=model_name,
                    prompt_text=prompt,
                    response_text="",
                    response=None,
                    status="error",
                    error_message=str(e),
                )
                raise

            try:
                rebuild_client()
            except Exception:
                pass

            wait_s = compute_backoff(attempt)
            print(f"Retry testo {attempt + 1}/{max_retries} dopo errore: {e}")
            print(f"Attendo {wait_s:.1f}s...")
            time.sleep(wait_s)

    raise last_err


def call_gemini_video_model(
    file_obj,
    prompt: str,
    model_name: str,
    max_retries: int = MAX_RETRIES_VIDEO,
    branch_name: str = "common",
    block_name: str = "video_block",
    window_id: Optional[int] = None,
) -> dict:
    response, text = call_gemini_video_model_raw(
        file_obj=file_obj,
        prompt=prompt,
        model_name=model_name,
        config_obj=GEN_CONFIG,
        max_retries=max_retries,
        branch_name=branch_name,
        block_name=block_name,
        window_id=window_id,
    )
    return safe_json(text)


def call_gemini_text_model(
    prompt: str,
    model_name: str,
    max_retries: int = MAX_RETRIES_TEXT,
    branch_name: str = "common",
    block_name: str = "text_block",
    window_id: Optional[int] = None,
) -> dict:
    response, text = call_gemini_text_model_raw(
        prompt=prompt,
        model_name=model_name,
        config_obj=GEN_CONFIG,
        max_retries=max_retries,
        branch_name=branch_name,
        block_name=block_name,
        window_id=window_id,
    )
    return safe_json(text)


def call_gemini_video(
    file_obj,
    prompt: str,
    max_retries: int = MAX_RETRIES_VIDEO,
    branch_name: str = "common",
    block_name: str = "video_block",
    window_id: Optional[int] = None,
) -> dict:
    try:
        return call_gemini_video_model(
            file_obj=file_obj,
            prompt=prompt,
            model_name=MODEL_NAME,
            max_retries=max_retries,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
        )
    except Exception as e:
        if FALLBACK_MODEL_NAME and is_retryable_error(e):
            print(f"Passo al modello fallback VIDEO dopo errore: {e}")
            return call_gemini_video_model(
                file_obj=file_obj,
                prompt=prompt,
                model_name=FALLBACK_MODEL_NAME,
                max_retries=max_retries,
                branch_name=branch_name,
                block_name=block_name,
                window_id=window_id,
            )
        raise


def call_gemini_text(
    prompt: str,
    max_retries: int = MAX_RETRIES_TEXT,
    branch_name: str = "common",
    block_name: str = "text_block",
    window_id: Optional[int] = None,
) -> dict:
    try:
        return call_gemini_text_model(
            prompt=prompt,
            model_name=MODEL_NAME,
            max_retries=max_retries,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
        )
    except Exception as e:
        if FALLBACK_MODEL_NAME and is_retryable_error(e):
            print(f"Passo al modello fallback TESTO dopo errore: {e}")
            return call_gemini_text_model(
                prompt=prompt,
                model_name=FALLBACK_MODEL_NAME,
                max_retries=max_retries,
                branch_name=branch_name,
                block_name=block_name,
                window_id=window_id,
            )
        raise


def call_gemini_video_raw(
    file_obj,
    prompt: str,
    max_retries: int = MAX_RETRIES_VIDEO,
    branch_name: str = "common",
    block_name: str = "video_block",
    window_id: Optional[int] = None,
) -> Tuple[Any, str]:
    try:
        return call_gemini_video_model_raw(
            file_obj=file_obj,
            prompt=prompt,
            model_name=MODEL_NAME,
            config_obj=GEN_CONFIG,
            max_retries=max_retries,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
        )
    except Exception as e:
        if FALLBACK_MODEL_NAME and is_retryable_error(e):
            print(f"Passo al modello fallback VIDEO raw dopo errore: {e}")
            return call_gemini_video_model_raw(
                file_obj=file_obj,
                prompt=prompt,
                model_name=FALLBACK_MODEL_NAME,
                config_obj=GEN_CONFIG,
                max_retries=max_retries,
                branch_name=branch_name,
                block_name=block_name,
                window_id=window_id,
            )
        raise


def call_gemini_text_raw(
    prompt: str,
    max_retries: int = MAX_RETRIES_TEXT,
    branch_name: str = "common",
    block_name: str = "text_block",
    window_id: Optional[int] = None,
    config_obj=None,
) -> Tuple[Any, str]:
    effective_config = GEN_CONFIG if config_obj is None else config_obj

    try:
        return call_gemini_text_model_raw(
            prompt=prompt,
            model_name=MODEL_NAME,
            config_obj=effective_config,
            max_retries=max_retries,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
        )
    except Exception as e:
        if FALLBACK_MODEL_NAME and is_retryable_error(e):
            print(f"Passo al modello fallback TESTO raw dopo errore: {e}")
            return call_gemini_text_model_raw(
                prompt=prompt,
                model_name=FALLBACK_MODEL_NAME,
                config_obj=effective_config,
                max_retries=max_retries,
                branch_name=branch_name,
                block_name=block_name,
                window_id=window_id,
            )
        raise


def call_gemini_video_until_valid_json(
    file_obj,
    prompt: str,
    branch_name: str,
    block_name: str,
    window_id: Optional[int] = None,
    max_generation_attempts: int = 5,
) -> dict:
    """
    Genera il JSON da video e, se il risultato è corrotto o non valido:
    - SCARTA l'output;
    - rigenera da zero finché ottiene un JSON valido.
    """
    last_text = ""

    for generation_attempt in range(1, max_generation_attempts + 1):
        print(
            f"🔁 [{branch_name} | {block_name}] Tentativo generazione JSON "
            f"{generation_attempt}/{max_generation_attempts}..."
        )

        _, raw_text = call_gemini_video_raw(
            file_obj=file_obj,
            prompt=prompt,
            max_retries=MAX_RETRIES_VIDEO,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
        )

        last_text = raw_text
        parsed = safe_json_or_none(
            raw_text,
            debug=(generation_attempt == max_generation_attempts),
        )
        if parsed is not None:
            print(
                f"✅ [{branch_name} | {block_name}] JSON valido ottenuto "
                f"al tentativo {generation_attempt}."
            )
            return parsed

        print(f"🗑️ [{branch_name} | {block_name}] Output non valido scartato.")
        time.sleep(1.0)

    raise RuntimeError(
        f"Impossibile ottenere un JSON valido per {branch_name}/{block_name} "
        f"dopo {max_generation_attempts} tentativi. Ultimo output visto: {last_text[:300]}"
    )


def call_gemini_text_until_valid_json(
    prompt: str,
    branch_name: str,
    block_name: str,
    window_id: Optional[int] = None,
    max_generation_attempts: int = 99,
) -> dict:
    """
    Genera JSON da prompt testuale e, se il risultato è corrotto o non valido:
    - SCARTA l'output;
    - rigenera da zero finché ottiene un JSON valido.
    """
    last_text = ""

    for generation_attempt in range(1, max_generation_attempts + 1):
        print(
            f"🔁 [{branch_name} | {block_name}] Tentativo generazione JSON "
            f"{generation_attempt}/{max_generation_attempts}..."
        )

        _, raw_text = call_gemini_text_raw(
            prompt=prompt,
            max_retries=MAX_RETRIES_TEXT,
            branch_name=branch_name,
            block_name=block_name,
            window_id=window_id,
            config_obj=GEN_CONFIG,
        )

        last_text = raw_text
        parsed = safe_json_or_none(
            raw_text,
            debug=(generation_attempt == max_generation_attempts),
        )
        if parsed is not None:
            print(
                f"✅ [{branch_name} | {block_name}] JSON valido ottenuto "
                f"al tentativo {generation_attempt}."
            )
            return parsed

        print(f"🗑️ [{branch_name} | {block_name}] Output non valido scartato.")
        time.sleep(1.0)

    raise RuntimeError(
        f"Impossibile ottenere un JSON valido per {branch_name}/{block_name} "
        f"dopo {max_generation_attempts} tentativi. Ultimo output visto: {last_text[:300]}"
    )


print("Utility Gemini configurate correttamente.")

## 7. Prompt comuni e ramo VLM libero

Dopo il **windowing temporale** la pipeline si separa nel primo livello di rami confrontabili. Il ramo **VLM libero** parte direttamente dalla clip della finestra, mentre i rami **VLM guidato** e **VLM guidato + AHP** continuano a condividere la base guidata composta da prompt descrittivo, question set diagnostico aggiornato e motore causale leggero.

Per il ramo libero restano due prompt dedicati:
- un prompt descrittivo-causale realmente più libero, con JSON valido ma senza schema rigido imposto come template di risposta;
- un prompt di scoring istantaneo separato che assegna un punteggio 0-100 e una breve motivazione.


In [ ]:

def build_prompt_descriptive(window: dict) -> str:
    schema = json.dumps({
        "scene_state": "",
        "environmental_conditions": [],
        "people": [],
        "objects": [],
        "machines": [],
        "observable_actions": [],
        "visible_ppe": [],
        "notes_visibility_limitations": ""
    }, indent=2, ensure_ascii=False)

    return f"""
Sei un assistente esperto di analisi video per ambienti industriali e logistici.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}
- duration_s: {window['duration']}

COMPITO
Analizza la clip video e descrivi in modo oggettivo e neutro la scena osservata.
NON fare valutazioni di rischio in questa fase. Descrivi SOLO ciò che è visibile.

ELEMENTI DA DESCRIVERE
1. Stato generale della scena.
2. Condizioni ambientali osservabili.
3. Persone presenti.
4. Oggetti presenti.
5. Macchinari presenti.
6. Azioni osservabili.
7. DPI visibili sulle persone presenti, se osservabili.
8. Limitazioni di visibilità, occlusioni o ambiguità.

FORMATO DI OUTPUT
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza markdown.
Schema atteso:
{schema}

LIMITE
Mantieni la risposta entro {MAX_CHARS_DIAGNOSTIC} caratteri totali.
""".strip()


def build_prompt_question_set(window: dict, scene_json: dict) -> str:
    scene_str = json.dumps(scene_json, indent=2, ensure_ascii=False)
    schema = json.dumps({
        "unsafe_behaviors": [],
        "vulnerable_elements": [],
        "risk_cues": {
            "S1_people_vehicle_interference": 0.0,
            "S2_visibility_attention_reduction": 0.0,
            "S3_non_compliant_behavior": 0.0,
            "S4_vehicle_infrastructure_interference": 0.0,
            "S5_storage_structural_instability": 0.0,
            "S6_access_to_moving_mechanical_parts": 0.0,
            "S7_manual_handling_heavy_unstable_loads": 0.0
        },
        "cue_confidence": 0.0,
        "diagnostic_notes": ""
    }, indent=2, ensure_ascii=False)

    return f"""
Sei un esperto di safety analysis per ambienti industriali e logistici.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}

DESCRIZIONE SCENA DA ANALISI PRECEDENTE
{scene_str}

COMPITO
Analizza la clip video insieme alla descrizione e valuta la safety della scena.
Devi attraversare una batteria di domande diagnostiche sulle 7 situazioni pericolose S1-S7 e restituire un solo valore aggregato per ciascuna categoria nel campo risk_cues.

BATTERIA DI DOMANDE DIAGNOSTICHE

S1. Interferenza tra persone e mezzi di movimentazione negli spazi comuni
1. Sono presenti nella stessa area operativa persone e mezzi di movimentazione, con condivisione dello spazio o dei percorsi?
2. La separazione tra percorso pedonale e traiettoria del mezzo è assente, insufficiente o non chiaramente rispettata?
3. Una o più persone risultano esposte al passaggio, alla manovra o alla vicinanza del mezzo nello spazio comune?

S2. Riduzione della visibilità durante lo svolgimento di un'attività
4. Sono presenti carichi, attrezzature, posture, orientamenti o condizioni operative che possono limitare il campo visivo o l'attenzione di un soggetto?
5. Questa riduzione di visibilità/attenzione, se presente, si manifesta durante una specifica attività o manovra operativa?
6. La limitazione di visibilità o attenzione, se presente, espone la persona a ostacoli, mezzi, parti in movimento o punti critici della scena?

S3. Comportamenti non conformi durante lo svolgimento di un'attività
7. Sono presenti comportamenti non conformi rispetto a procedure, percorsi pedonali o modalità operative previste?
8. Il comportamento osservato è legato a distrazione, fretta o mancato rispetto delle istruzioni?
9. Una o più persone risultano esposte a un rischio concreto per effetto di tale comportamento?

S4. Interferenza tra mezzo e infrastrutture interne all'impianto
10. È presente un mezzo che transita o manovra in prossimità di infrastrutture interne all'impianto?
11. Sono assenti o insufficienti distanze di sicurezza, protezioni o spazi di manovra adeguati?
12. Una o più persone risultano esposte a urto, collisione o schiacciamento?

S5. Instabilità di elementi strutturali o di stoccaggio soggetti a urto o sovraccarico
13. La scena osservata è un'area di stoccaggio o magazzino, con scaffalature, cataste, depositi o carichi organizzati in quota?
14. La configurazione osservata mostra condizioni compatibili con elementi di stoccaggio instabili, sovraccarichi o suscettibili di caduta del materiale, oppure con un possibile impatto sulla scaffalatura o su altri elementi di stoccaggio?
15. La configurazione osservata espone persone presenti al possibile coinvolgimento da caduta di materiale, collasso o spostamento del carico?

S6. Accesso a organi meccanici in movimento durante operazioni manuali
16. È presente un intervento manuale su una macchina o su organi meccanici in movimento, come regolazione, sblocco, pulizia o estrazione?
17. Sono visibili misure di protezione o isolamento che impediscono il contatto diretto con gli organi meccanici in movimento?
18. Sono presenti condizioni operative che espongono l'operatore a organi meccanici in movimento durante l'intervento manuale?

S7. Movimentazione manuale di carichi pesanti o instabili senza ausili meccanici adeguati
19. È presente una movimentazione manuale di un carico pesante, instabile o difficilmente gestibile?
20. Sono assenti, non utilizzati o non disponibili ausili meccanici che sarebbero appropriati per la movimentazione osservata?
21. Una o più persone sono esposte a caduta del carico, schiacciamento o sovraccarico biomeccanico durante la movimentazione?

ISTRUZIONI DI VALUTAZIONE
- Per ciascuna situazione S1-S7, integra le tre domande in un solo valore continuo tra 0.0 e 1.0 nel campo risk_cues.
- Usa valori vicini a 0 se la situazione non è supportata dagli elementi osservabili.
- Usa valori intermedi se la situazione è solo parzialmente supportata o ambigua.
- Usa valori alti se fonte/configurazione, insufficienza delle misure di riduzione ed esposizione risultano chiaramente presenti.
- Basa la valutazione solo su elementi visibili nella clip o coerenti con la descrizione fornita.
- Non inventare dettagli non osservabili.

SCALA DA USARE
- 0.0 assente
- 0.25 debole evidenza
- 0.5 presenza moderata o ambigua ma plausibile
- 0.75 forte evidenza
- 1.0 critica o molto evidente

IMPORTANTE
- NON restituire un punteggio per ogni domanda.
- Restituisci un solo valore aggregato per ciascuna categoria S1-S7.
- Non trattare la mancata visibilità come prova di assenza.
- Se una categoria non è valutabile con sufficiente visibilità, assegna un valore prudente basso e spiega il limite in diagnostic_notes.
- Valuta le situazioni pericolose osservabili, non introdurre cause non visibili né dettagli non supportati.

Compila inoltre:
- unsafe_behaviors
- vulnerable_elements
- cue_confidence (0.0-1.0)
- diagnostic_notes

FORMATO DI OUTPUT
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza markdown.
Schema atteso:
{schema}

LIMITE
Mantieni la risposta entro {MAX_CHARS_QUESTION_SET} caratteri totali.
""".strip()


def build_prompt_causal(window: dict, scene_json: dict, diagnostic_json: dict) -> str:
    scene_str = json.dumps(scene_json, indent=2, ensure_ascii=False)
    diagnostic_str = json.dumps(diagnostic_json, indent=2, ensure_ascii=False)
    schema = json.dumps({
        "causal_chain_plausible": False,
        "causal_chain_text": "",
        "near_miss_or_incident_scenario": "",
        "contributing_factors": []
    }, indent=2, ensure_ascii=False)

    return f"""
Sei un esperto di analisi del rischio e safety industriale.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}

DESCRIZIONE SCENA
{scene_str}

CUE DI RISCHIO IDENTIFICATI
{diagnostic_str}

COMPITO
Ragiona in modo prudente sulle relazioni causa-effetto osservabili.
Basa il ragionamento SOLO sugli elementi visibili o già descritti. Non ipotizzare fattori non osservati.

Devi rispondere a:
1. La configurazione osservata può portare a un near miss o incidente? true/false
2. Se sì, qual è la catena causale plausibile?
3. Qual è lo scenario di near miss o incidente più probabile?
4. Quali fattori contribuiscono al rischio?

FORMATO DI OUTPUT
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza markdown.
Schema atteso:
{schema}

LIMITE
Mantieni la risposta entro {MAX_CHARS_CAUSAL} caratteri totali.
""".strip()


def build_prompt_free_descriptive_causal(window: dict) -> str:
    return f"""
Sei un esperto di safety analysis per video in contesti industriali e logistici.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}
- duration_s: {window['duration']}

COMPITO
Analizza la clip video e restituisci una descrizione libera ma strutturata della finestra temporale.
L'output deve essere utile per una safety analysis industriale o logistica, ma senza imporre un question set né una lista rigida di chiavi da copiare.

ASPETTI CHE DEVONO EMERGERE NEL CONTENUTO JSON
1. Descrizione generale della scena.
2. Contesto industriale o logistico osservabile.
3. Persone presenti.
4. Oggetti presenti.
5. Macchinari presenti.
6. Azioni osservabili.
7. Condizioni ambientali osservabili.
8. Limiti di visibilità, occlusioni o ambiguità.
9. Ragionamento causale prudente su dinamiche in corso.
10. Possibili sviluppi rischiosi plausibili (near miss o incidente), solo se supportati da elementi osservabili.
11. Fattori contribuenti osservabili.
12. DPI utilizzati

VINCOLI EPISTEMICI
- Basati solo su elementi visibili nella clip.
- Non inventare cause, intenzioni, stati interni o fattori non osservabili.
- Se un elemento è dubbio, dichiaralo esplicitamente come dubbio, ambiguità o limite osservativo.
- I possibili sviluppi rischiosi devono essere prudenziali e motivati da segnali osservabili già menzionati.

FORMATO DI OUTPUT
- Rispondi esclusivamente con JSON valido.
- Non usare markdown.
- Non aggiungere testo fuori dal JSON.
- Organizza autonomamente la struttura del JSON nel modo che ritieni più coerente e leggibile.
- Il JSON può usare chiavi, sotto-sezioni, liste e testi sintetici liberamente scelti, purché i contenuti richiesti siano chiaramente presenti.

LIMITE
Mantieni la risposta entro {MAX_CHARS_FREE_DESCRIPTIVE} caratteri totali.
""".strip()


def build_prompt_free_scoring(window: dict, free_scene_json: dict) -> str:

    free_scene_str = json.dumps(free_scene_json, indent=2, ensure_ascii=False)
    schema = json.dumps({
        "instantaneous_risk_score": 0,
        "score_rationale": ""
    }, indent=2, ensure_ascii=False)

    return f"""
Sei un responsabile della sicurezza in un ambiente industriale e logistico.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}

INPUT DESCRITTIVO-CAUSALE DEL RAMO VLM LIBERO
{free_scene_str}

COMPITO
Assegna un instantaneous_risk_score da 0 a 100 sulla base del contenuto fornito.
NON assegnare il colore del semaforo.

Scala orientativa:
- 0-24 rischio basso
- 25-49 rischio moderato
- 50-74 rischio significativo
- 75-100 rischio elevato/critico

VINCOLI IMPORTANTI
- Basa lo score solo sugli elementi osservabili o già descritti.
- Non introdurre fattori nuovi non supportati dal contenuto disponibile.
- Fornisci anche una breve motivazione del punteggio.
- Non assegnare il semaforo: smoothing e semaforo restano nel codice.

FORMATO DI OUTPUT
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza markdown.
Schema atteso:
{schema}

LIMITE
Mantieni la risposta entro {MAX_CHARS_FREE_SCORING} caratteri totali.
""".strip()


## 8. Prompt scoring istantaneo VLM

Questo prompt è usato **solo nel ramo VLM**. Il modello assegna il punteggio istantaneo 0-100 e una breve motivazione, ma **non** assegna il colore del semaforo. Il semaforo resta sempre calcolato dal codice dopo lo score dinamico [file:299].

In [ ]:
def build_prompt_vlm_scoring(window: dict, scene_json: dict, diagnostic_json: dict, causal_json: dict) -> str:
    scene_str = json.dumps(scene_json, indent=2, ensure_ascii=False)
    diagnostic_str = json.dumps(diagnostic_json, indent=2, ensure_ascii=False)
    causal_str = json.dumps(causal_json, indent=2, ensure_ascii=False)
    schema = json.dumps({
        "risk_score_0_100": 0,
        "score_rationale": ""
    }, indent=2, ensure_ascii=False)

    return f"""
Sei un responsabile della sicurezza in un ambiente industriale e logistico.

LINGUA DI OUTPUT: {OUTPUT_LANGUAGE}

FINESTRA TEMPORALE
- start_time_s: {window['start']}
- end_time_s: {window['end']}

DESCRIZIONE SCENA
{scene_str}

CUE DI RISCHIO
{diagnostic_str}

ANALISI CAUSALE
{causal_str}

COMPITO
Sulla base di tutte le informazioni precedenti, assegna un PUNTEGGIO DI RISCHIO istantaneo da 0 a 100.
NON assegnare il colore del semaforo.

Scala orientativa:
- 0-24 rischio basso
- 25-49 rischio moderato
- 50-74 rischio significativo
- 75-100 rischio elevato/critico

Fornisci anche una breve motivazione del punteggio (massimo 2 frasi).

FORMATO DI OUTPUT
Rispondi ESCLUSIVAMENTE con JSON valido, senza testo aggiuntivo, senza markdown.
Schema atteso:
{schema}

LIMITE
Mantieni la risposta entro {MAX_CHARS_DECISION_VLM} caratteri totali.
""".strip()

## 9. Definizione pesi AHP e funzioni monotone esogene

Il ramo AHP riusa la logica del notebook più recente: pesi AHP fissati a monte e funzioni monotone specifiche per ciascun rischio. Il VLM produce i cue osservativi; il codice li trasforma e li aggrega in uno score interpretabile [file:300].

In [ ]:
def clamp_01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def clamp_score_0_100(x: float) -> float:
    return round(max(0.0, min(100.0, float(x))), 2)


def sigmoid(x: float, k: float, tau: float) -> float:
    x = clamp_01(x)
    return float(1.0 / (1.0 + math.exp(-k * (x - tau))))


def concave_increasing(x: float, gamma: float = 3.0) -> float:
    x = clamp_01(x)
    if gamma <= 0:
        raise ValueError(f"gamma deve essere > 0, ricevuto {gamma}")
    return float((1.0 - math.exp(-gamma * x)) / (1.0 - math.exp(-gamma)))


def convex_increasing(x: float, gamma: float = 2.0) -> float:
    x = clamp_01(x)
    if gamma <= 0:
        raise ValueError(f"gamma deve essere > 0, ricevuto {gamma}")
    return float(x ** gamma)


def f_s1(x: float) -> float:
    # S1: lineare
    return clamp_01(x)


def f_s2(x: float) -> float:
    # S2: crescente a tassi decrescenti
    return clamp_01(concave_increasing(x, GAMMA_S2))


def f_s3(x: float) -> float:
    # S3: sigmoide
    return clamp_01(sigmoid(x, K_S3, TAU_S3))


def f_s4(x: float) -> float:
    # S4: sigmoide
    return clamp_01(sigmoid(x, K_S4, TAU_S4))


def f_s5(x: float) -> float:
    # S5: crescente a tassi crescenti
    return clamp_01(convex_increasing(x, GAMMA_S5))


def f_s6(x: float) -> float:
    # S6: sigmoide
    return clamp_01(sigmoid(x, K_S6, TAU_S6))


def f_s7(x: float) -> float:
    # S7: crescente a tassi crescenti
    return clamp_01(convex_increasing(x, GAMMA_S7))


def transform_risk_value(risk_code: str, raw_value: float) -> float:
    dispatch = {
        "S1": f_s1,
        "S2": f_s2,
        "S3": f_s3,
        "S4": f_s4,
        "S5": f_s5,
        "S6": f_s6,
        "S7": f_s7,
    }
    if risk_code not in dispatch:
        raise KeyError(f"Situazione pericolosa non riconosciuta: {risk_code}")
    return round(dispatch[risk_code](raw_value), 4)


ahp_df = pd.DataFrame([
    {"risk_code": k, "risk_label": RISK_LABELS[k], "weight": v}
    for k, v in AHP_WEIGHTS.items()
]).sort_values("risk_code")

print("Somma pesi AHP:", round(ahp_df["weight"].sum(), 6))
ahp_df

## 10. Scoring istantaneo AHP-based

Lo score istantaneo AHP viene calcolato nel codice usando i cue del question set, le funzioni monotone e i pesi AHP. Se `USE_CUE_CONFIDENCE = True`, lo score viene modulato dalla confidenza complessiva del VLM [file:300].

In [ ]:
def extract_raw_risk_values(diagnostic_json: dict) -> dict:
    risk_cues = diagnostic_json.get("risk_cues", {}) if isinstance(diagnostic_json, dict) else {}
    raw = {}
    for risk_code, cue_key in RISK_CUE_KEYS.items():
        try:
            raw[risk_code] = clamp_01(float(risk_cues.get(cue_key, 0.0)))
        except Exception:
            raw[risk_code] = 0.0
    return raw


def compute_ahp_risk_contributions(diagnostic_json: dict, use_confidence: bool = USE_CUE_CONFIDENCE) -> dict:
    raw = extract_raw_risk_values(diagnostic_json)
    try:
        cue_confidence = clamp_01(float(diagnostic_json.get("cue_confidence", 0.0)))
    except Exception:
        cue_confidence = 0.0

    confidence_factor = cue_confidence if use_confidence else 1.0
    transformed = {r: transform_risk_value(r, raw[r]) for r in raw}
    contributions = {
        r: round(100.0 * AHP_WEIGHTS[r] * transformed[r] * confidence_factor, 4)
        for r in raw
    }
    instantaneous_score = round(sum(contributions.values()), 2)

    if instantaneous_score > 0:
        contribution_pct = {
            r: round(contributions[r] / instantaneous_score * 100.0, 2)
            for r in raw
        }
    else:
        contribution_pct = {r: 0.0 for r in raw}

    return {
        "raw": raw,
        "transformed": transformed,
        "weights": dict(AHP_WEIGHTS),
        "contributions": contributions,
        "contribution_pct": contribution_pct,
        "cue_confidence": cue_confidence,
        "confidence_factor_applied": confidence_factor,
        "instantaneous_risk_score": instantaneous_score,
    }

example_diagnostic = {
    "risk_cues": {
        "S1_people_vehicle_interference": 0.75,
        "S2_visibility_attention_reduction": 0.50,
        "S3_non_compliant_behavior": 0.25,
        "S4_vehicle_infrastructure_interference": 0.75,
        "S5_storage_structural_instability": 0.50,
        "S6_access_to_moving_mechanical_parts": 0.50,
        "S7_manual_handling_heavy_unstable_loads": 0.25,
    },
    "cue_confidence": 0.8,
}
compute_ahp_risk_contributions(example_diagnostic)

## 11. Smoothing temporale dei tre rami

Tutti e tre i rami utilizzano uno **score dinamico** calcolato dal codice. La formulazione adottata è la seguente:

$$
score_{dyn}(t)
=
\alpha \, score_{dyn}(t-1)
+
(1-\alpha) \, score_{inst}(t)
+
\beta \, \max \left(0,\ score_{inst}(t)-score_{inst}(t-1)\right)
$$

In questa formulazione:

- `score_inst(t)` è lo score istantaneo della finestra corrente;
- `score_dyn(t-1)` è lo score dinamico della finestra precedente;
- `alpha` controlla il peso attribuito alla memoria temporale;
- `beta` controlla l'intensità della risposta agli aumenti improvvisi del rischio;
- il termine `max(0, score_inst(t)-score_inst(t-1))` si attiva solo quando il rischio istantaneo cresce rispetto alla finestra precedente.

Questa scelta consente al sistema di essere più stabile nel tempo, ma anche più reattivo quando emerge un incremento improvviso del rischio.

Il ramo VLM guidato usa come `score_inst` il valore assegnato dal modello; il ramo VLM guidato + AHP usa il valore calcolato dal codice; il ramo VLM libero usa il valore assegnato dal proprio blocco di scoring libero [file:299][file:300].

In [ ]:
class DynamicRiskSmoother:
    def __init__(self, alpha: float, beta: float):
        if not 0.0 <= alpha <= 1.0:
            raise ValueError(f"alpha deve essere in [0,1], ricevuto {alpha}")
        if beta < 0.0:
            raise ValueError(f"beta deve essere >= 0, ricevuto {beta}")
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.prev_dynamic = None
        self.prev_instantaneous = None

    def update(self, instantaneous_score: float) -> float:
        st = clamp_score_0_100(instantaneous_score)
        if self.prev_dynamic is None or self.prev_instantaneous is None:
            rt = st
        else:
            escalation_term = self.beta * max(0.0, st - self.prev_instantaneous)
            rt = self.alpha * self.prev_dynamic + (1.0 - self.alpha) * st + escalation_term
        rt = clamp_score_0_100(rt)
        self.prev_dynamic = rt
        self.prev_instantaneous = st
        return round(rt, 2)

    def reset(self):
        self.prev_dynamic = None
        self.prev_instantaneous = None

smoother_vlm = DynamicRiskSmoother(alpha=ALPHA_VLM, beta=BETA_VLM)
smoother_ahp = DynamicRiskSmoother(alpha=ALPHA_AHP, beta=BETA_AHP)
smoother_vlm_free = DynamicRiskSmoother(alpha=ALPHA_VLM_FREE, beta=BETA_VLM_FREE)
print("Smoother VLM guidato, AHP e VLM libero inizializzati.")


## 12. Assegnazione semaforo nei tre rami

Il colore del semaforo viene assegnato **solo** dopo il calcolo dello score dinamico. La funzione è comune, ma viene applicata separatamente ai tre rami.

In [ ]:
def score_to_traffic_light(score: float) -> str:
    score = float(score)
    if score <= THRESH_GREEN[1]:
        return "verde"
    elif score <= THRESH_YELLOW[1]:
        return "giallo"
    elif score <= THRESH_ORANGE[1]:
        return "arancione"
    return "rosso"

for val in [10, 35, 60, 80]:
    print(val, score_to_traffic_light(val))

## 13. Esecuzione pipeline ibrida quasi parallela

Per ogni finestra temporale l'ordine di esecuzione resta **conservativo**, ma con la correzione architetturale richiesta: dopo il blocco di **windowing temporale** parte subito il ramo **VLM libero**, mentre i rami **VLM guidato** e **VLM guidato + AHP** proseguono con i tre blocchi guidati comuni.

In questo modo la pipeline mantiene una parte iniziale comune minima, una prima divisione metodologica tra ramo libero e ramo guidato, e una seconda divisione tra i due rami guidati.


In [ ]:
records = []

EMOJI = {
    "start": "🚀",
    "window": "🪟",
    "clip": "✂️",
    "upload": "⤴️",
    "ready": "✅",
    "free_desc": "🟢",
    "free_score": "🟩",
    "free_smooth": "📈",
    "free_traffic": "🚦",
    "prompt_desc": "📝",
    "prompt_qs": "📋",
    "prompt_causal": "🧠",
    "guided_score": "🔵",
    "guided_smooth": "📈",
    "guided_traffic": "🚦",
    "ahp_score": "🟠",
    "ahp_smooth": "📈",
    "ahp_traffic": "🚦",
    "error": "❌",
    "done": "🏁",
    "delete": "🗑️",
}

TRAFFIC_EMOJI = {
    "verde": "🟢",
    "giallo": "🟡",
    "arancione": "🟠",
    "rosso": "🔴",
    "disabilitato": "⚪",
}

def cleanup_local_clip_with_retry(path, retries=6, wait_seconds=1.0):
    if path is None:
        return
    try:
        exists_now = path.exists()
    except Exception:
        exists_now = False

    if not exists_now:
        return

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            path.unlink()
            return
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(wait_seconds)
            else:
                print(f"Cleanup clip locale non riuscito: {e}")

if not video_path.exists():
    print(f"{EMOJI['error']} Video non disponibile. Aggiorna VIDEO_PATH e riesegui la pipeline completa.")
else:
    print(f"{EMOJI['start']} Avvio pipeline ibrida: {len(windows)} finestre da analizzare")
    print("=" * 80)

    for w in tqdm(windows, desc="Pipeline ibrida in corso"):
        wid, wstart, wend = w["id"], w["start"], w["end"]

        print("=" * 80)
        print(f"{EMOJI['window']} Finestra {wid:03d} | {wstart:.2f}s - {wend:.2f}s")

        clip_path = CLIPS_DIR / f"window_{wid:04d}_{wstart:.2f}_{wend:.2f}.mp4"
        remote_file = None
        record = None

        branch_calls_before = dict(_BRANCH_API_CALL_COUNTERS)

        try:
            extract_clip(video_path, clip_path, wstart, wend)
            print(f"{EMOJI['clip']} Clip estratta: {clip_path.name}")

            print(f"{EMOJI['upload']} Upload clip su Gemini Files API...")
            remote_file = upload_clip(clip_path)
            remote_state = getattr(remote_file.state, "name", str(remote_file.state))
            print(f"{EMOJI['ready']} File pronto: {remote_file.name} | stato={remote_state}")

            # =========================================================
            # PRIMA BIFORCAZIONE: RAMO VLM LIBERO
            # =========================================================
            if ENABLE_VLM_FREE_BRANCH:
                print(f"{EMOJI['free_desc']} [Ramo VLM libero] Blocco descrittivo-causale...")
                free_scene = call_gemini_video_until_valid_json(
                    file_obj=remote_file,
                    prompt=build_prompt_free_descriptive_causal(w),
                    branch_name="vlm_free",
                    block_name="vlm_free_descriptive_causal",
                    window_id=wid,
                    max_generation_attempts=99,
                )
                free_scene_str = json.dumps(free_scene, ensure_ascii=False)

                print(f"{EMOJI['free_score']} [Ramo VLM libero] Scoring istantaneo...")
                free_decision = call_gemini_text_until_valid_json(
                    prompt=build_prompt_free_scoring(w, free_scene),
                    branch_name="vlm_free",
                    block_name="vlm_free_scoring",
                    window_id=wid,
                    max_generation_attempts=99,
                )

                instantaneous_vlm_free = clamp_score_0_100(
                    float(
                        free_decision.get(
                            "instantaneous_risk_score",
                            free_decision.get("risk_score_0_100", 0.0)
                        )
                    )
                )
                rationale_vlm_free = free_decision.get("score_rationale", "")
                print(f"{EMOJI['free_score']} Score istantaneo VLM libero: {instantaneous_vlm_free:.2f}")

                print(f"{EMOJI['free_smooth']} [Ramo VLM libero] Smoothing temporale...")
                dynamic_vlm_free = smoother_vlm_free.update(instantaneous_vlm_free)
                print(f"{EMOJI['free_smooth']} Dynamic score VLM libero: {dynamic_vlm_free:.2f}")

                print(f"{EMOJI['free_traffic']} [Ramo VLM libero] Assegnazione semaforo...")
                traffic_light_vlm_free = score_to_traffic_light(dynamic_vlm_free)
                print(f"{TRAFFIC_EMOJI.get(traffic_light_vlm_free, '⚪')} Semaforo VLM libero: {traffic_light_vlm_free.upper()}")
            else:
                free_scene = {}
                free_scene_str = json.dumps({}, ensure_ascii=False)
                free_decision = {}
                instantaneous_vlm_free = np.nan
                rationale_vlm_free = "Ramo VLM libero disabilitato"
                dynamic_vlm_free = np.nan
                traffic_light_vlm_free = "disabilitato"
                print(f"{EMOJI['free_desc']} [Ramo VLM libero] Disabilitato")

            # =========================================================
            # BACKBONE GUIDATO COMUNE
            # =========================================================
            ENABLE_GUIDED_SHARED_BACKBONE = ENABLE_VLM_BRANCH or ENABLE_AHP_BRANCH

            if ENABLE_GUIDED_SHARED_BACKBONE:
                print(f"{EMOJI['prompt_desc']} [Parte guidata comune] Prompt descrittivo VLM...")
                scene = call_gemini_video_until_valid_json(
                    file_obj=remote_file,
                    prompt=build_prompt_descriptive(w),
                    branch_name="common",
                    block_name="guided_descriptive_prompt",
                    window_id=wid,
                    max_generation_attempts=99,
                )
                scene_str = json.dumps(scene, ensure_ascii=False)

                print(f"{EMOJI['prompt_qs']} [Parte guidata comune] Question set diagnostico VLM...")
                diagnostic = call_gemini_text_until_valid_json(
                    prompt=build_prompt_question_set(w, scene),
                    branch_name="common",
                    block_name="guided_question_set",
                    window_id=wid,
                    max_generation_attempts=99,
                )
                diagnostic_str = json.dumps(diagnostic, ensure_ascii=False)

                print(f"{EMOJI['prompt_causal']} [Parte guidata comune] Motore causale leggero VLM...")
                causal = call_gemini_text_until_valid_json(
                    prompt=build_prompt_causal(w, scene, diagnostic),
                    branch_name="common",
                    block_name="guided_causal_engine",
                    window_id=wid,
                    max_generation_attempts=99,
                )
                causal_str = json.dumps(causal, ensure_ascii=False)
            else:
                print("[Parte guidata comune] Saltata: ENABLE_VLM_BRANCH=False e ENABLE_AHP_BRANCH=False")
                scene = {
                    "status": "skipped",
                    "reason": "Guided shared backbone disabled because both ENABLE_VLM_BRANCH and ENABLE_AHP_BRANCH are False"
                }
                diagnostic = {
                    "status": "skipped",
                    "reason": "Guided shared backbone disabled because both ENABLE_VLM_BRANCH and ENABLE_AHP_BRANCH are False",
                    "risk_cues": {},
                    "cue_confidence": 0.0,
                    "diagnostic_notes": "Parte guidata comune non eseguita perché sia ENABLE_VLM_BRANCH sia ENABLE_AHP_BRANCH sono False."
                }
                causal = {
                    "status": "skipped",
                    "reason": "Guided shared backbone disabled because both ENABLE_VLM_BRANCH and ENABLE_AHP_BRANCH are False",
                    "causal_chain_plausible": False,
                    "causal_chain_text": "Parte guidata comune non eseguita.",
                    "near_miss_or_incident_scenario": "",
                    "contributing_factors": []
                }
                scene_str = json.dumps(scene, ensure_ascii=False)
                diagnostic_str = json.dumps(diagnostic, ensure_ascii=False)
                causal_str = json.dumps(causal, ensure_ascii=False)

            # =========================================================
            # SECONDA BIFORCAZIONE: RAMO VLM GUIDATO
            # =========================================================
            if ENABLE_VLM_BRANCH:
                print(f"{EMOJI['guided_score']} [Ramo VLM guidato] Scoring istantaneo VLM...")
                vlm_decision = call_gemini_text_until_valid_json(
                    prompt=build_prompt_vlm_scoring(w, scene, diagnostic, causal),
                    branch_name="vlm_guided",
                    block_name="vlm_guided_scoring",
                    window_id=wid,
                    max_generation_attempts=99,
                )

                instantaneous_vlm = clamp_score_0_100(
                    float(
                        vlm_decision.get(
                            "risk_score_0_100",
                            vlm_decision.get("instantaneous_risk_score", 0.0)
                        )
                    )
                )
                rationale_vlm = vlm_decision.get("score_rationale", "")
                print(f"{EMOJI['guided_score']} Score istantaneo VLM guidato: {instantaneous_vlm:.2f}")

                print(f"{EMOJI['guided_smooth']} [Ramo VLM guidato] Smoothing temporale...")
                dynamic_vlm = smoother_vlm.update(instantaneous_vlm)
                print(f"{EMOJI['guided_smooth']} Dynamic score VLM guidato: {dynamic_vlm:.2f}")

                print(f"{EMOJI['guided_traffic']} [Ramo VLM guidato] Assegnazione semaforo...")
                traffic_light_vlm = score_to_traffic_light(dynamic_vlm)
                print(f"{TRAFFIC_EMOJI.get(traffic_light_vlm, '⚪')} Semaforo VLM guidato: {traffic_light_vlm.upper()}")

                vlm_decision_str = json.dumps(vlm_decision, ensure_ascii=False)
            else:
                vlm_decision = {}
                vlm_decision_str = json.dumps({}, ensure_ascii=False)
                instantaneous_vlm = np.nan
                rationale_vlm = "Ramo VLM guidato disabilitato"
                dynamic_vlm = np.nan
                traffic_light_vlm = "disabilitato"
                print(f"{EMOJI['guided_score']} [Ramo VLM guidato] Disabilitato")

            # =========================================================
            # SECONDA BIFORCAZIONE: RAMO VLM GUIDATO + AHP
            # =========================================================
            if ENABLE_AHP_BRANCH:
                print(f"{EMOJI['ahp_score']} [Ramo VLM guidato + AHP] Scoring istantaneo AHP-based...")
                ahp_pack = compute_ahp_risk_contributions(
                    diagnostic,
                    use_confidence=USE_CUE_CONFIDENCE
                )
                instantaneous_ahp = ahp_pack["instantaneous_risk_score"]
                print(f"{EMOJI['ahp_score']} Score istantaneo AHP: {instantaneous_ahp:.2f}")

                print(f"{EMOJI['ahp_smooth']} [Ramo VLM guidato + AHP] Smoothing temporale...")
                dynamic_ahp = smoother_ahp.update(instantaneous_ahp)
                print(f"{EMOJI['ahp_smooth']} Dynamic score AHP: {dynamic_ahp:.2f}")

                print(f"{EMOJI['ahp_traffic']} [Ramo VLM guidato + AHP] Assegnazione semaforo...")
                traffic_light_ahp = score_to_traffic_light(dynamic_ahp)
                print(f"{TRAFFIC_EMOJI.get(traffic_light_ahp, '⚪')} Semaforo AHP: {traffic_light_ahp.upper()}")
            else:
                ahp_pack = compute_ahp_risk_contributions(
                    {"risk_cues": {}, "cue_confidence": 0.0},
                    use_confidence=False
                )
                instantaneous_ahp = np.nan
                dynamic_ahp = np.nan
                traffic_light_ahp = "disabilitato"
                print(f"{EMOJI['ahp_score']} [Ramo VLM guidato + AHP] Disabilitato")

            branch_calls_after = dict(_BRANCH_API_CALL_COUNTERS)

            record = {
                "window_id": wid,
                "start_time_s": wstart,
                "end_time_s": wend,
                "window_duration_s": round(w["duration"], 3),

                "free_descriptive_causal_output_json": free_scene_str,
                "vlm_free_scoring_json": json.dumps(
                    {
                        "instantaneous_risk_score": None if pd.isna(instantaneous_vlm_free) else float(instantaneous_vlm_free),
                        "score_rationale": rationale_vlm_free,
                    },
                    ensure_ascii=False,
                ),
                "instantaneous_risk_score_vlm_free": instantaneous_vlm_free,
                "score_rationale_vlm_free": rationale_vlm_free,
                "dynamic_risk_score_vlm_free": dynamic_vlm_free,
                "traffic_light_color_vlm_free": traffic_light_vlm_free,

                "descriptive_prompt_output_json": scene_str,
                "question_set_prompt_output_json": diagnostic_str,
                "causal_prompt_output_json": causal_str,
                "vlm_guided_scoring_json": vlm_decision_str,

                "instantaneous_risk_score_vlm": instantaneous_vlm,
                "score_rationale_vlm": rationale_vlm,
                "dynamic_risk_score_vlm": dynamic_vlm,
                "traffic_light_color_vlm": traffic_light_vlm,

                "instantaneous_risk_score_ahp": instantaneous_ahp,
                "dynamic_risk_score_ahp": dynamic_ahp,
                "traffic_light_color_ahp": traffic_light_ahp,

                "cue_confidence": ahp_pack["cue_confidence"],
                "diagnostic_notes": diagnostic.get("diagnostic_notes", ""),

                "api_calls_common": branch_calls_after.get("common", 0) - branch_calls_before.get("common", 0),
                "api_calls_vlm_guided": branch_calls_after.get("vlm_guided", 0) - branch_calls_before.get("vlm_guided", 0),
                "api_calls_ahp": branch_calls_after.get("ahp", 0) - branch_calls_before.get("ahp", 0),
                "api_calls_vlm_free": branch_calls_after.get("vlm_free", 0) - branch_calls_before.get("vlm_free", 0),
            }

            for risk_code in RISK_CUE_KEYS.keys():
                record[f"{risk_code}_raw"] = ahp_pack["raw"][risk_code]
                record[f"{risk_code}_transformed"] = ahp_pack["transformed"][risk_code]
                record[f"{risk_code}_weight"] = ahp_pack["weights"][risk_code]
                record[f"{risk_code}_contribution"] = ahp_pack["contributions"][risk_code]
                record[f"{risk_code}_contribution_pct"] = ahp_pack["contribution_pct"][risk_code]

        except Exception as e:
            print(f"{EMOJI['error']} Errore nella finestra {wid}: {e}")

            fallback_pack = compute_ahp_risk_contributions(
                {"risk_cues": {}, "cue_confidence": 0.0},
                use_confidence=False
            )

            instantaneous_vlm = 0.0 if ENABLE_VLM_BRANCH else np.nan
            dynamic_vlm = smoother_vlm.update(instantaneous_vlm) if ENABLE_VLM_BRANCH else np.nan
            traffic_light_vlm = score_to_traffic_light(dynamic_vlm) if ENABLE_VLM_BRANCH else "disabilitato"

            instantaneous_ahp = 0.0 if ENABLE_AHP_BRANCH else np.nan
            dynamic_ahp = smoother_ahp.update(instantaneous_ahp) if ENABLE_AHP_BRANCH else np.nan
            traffic_light_ahp = score_to_traffic_light(dynamic_ahp) if ENABLE_AHP_BRANCH else "disabilitato"

            instantaneous_vlm_free = 0.0 if ENABLE_VLM_FREE_BRANCH else np.nan
            dynamic_vlm_free = smoother_vlm_free.update(instantaneous_vlm_free) if ENABLE_VLM_FREE_BRANCH else np.nan
            traffic_light_vlm_free = score_to_traffic_light(dynamic_vlm_free) if ENABLE_VLM_FREE_BRANCH else "disabilitato"

            branch_calls_after = dict(_BRANCH_API_CALL_COUNTERS)

            record = {
                "window_id": wid,
                "start_time_s": wstart,
                "end_time_s": wend,
                "window_duration_s": round(w["duration"], 3),

                "free_descriptive_causal_output_json": json.dumps({"error": f"ERRORE: {e}"}, ensure_ascii=False),
                "vlm_free_scoring_json": json.dumps({"error": f"ERRORE: {e}"}, ensure_ascii=False),
                "instantaneous_risk_score_vlm_free": instantaneous_vlm_free,
                "score_rationale_vlm_free": f"Errore pipeline: {e}",
                "dynamic_risk_score_vlm_free": dynamic_vlm_free,
                "traffic_light_color_vlm_free": traffic_light_vlm_free,

                "descriptive_prompt_output_json": json.dumps({"scene_state": f"ERRORE: {e}"}, ensure_ascii=False),
                "question_set_prompt_output_json": json.dumps({"risk_cues": {}, "cue_confidence": 0.0}, ensure_ascii=False),
                "causal_prompt_output_json": json.dumps(
                    {"causal_chain_plausible": False, "causal_chain_text": str(e)},
                    ensure_ascii=False
                ),
                "vlm_guided_scoring_json": json.dumps({"error": f"ERRORE: {e}"}, ensure_ascii=False),

                "instantaneous_risk_score_vlm": instantaneous_vlm,
                "score_rationale_vlm": f"Errore pipeline: {e}" if ENABLE_VLM_BRANCH else "Ramo VLM guidato disabilitato",
                "dynamic_risk_score_vlm": dynamic_vlm,
                "traffic_light_color_vlm": traffic_light_vlm,

                "instantaneous_risk_score_ahp": instantaneous_ahp,
                "dynamic_risk_score_ahp": dynamic_ahp,
                "traffic_light_color_ahp": traffic_light_ahp,

                "cue_confidence": fallback_pack["cue_confidence"],
                "diagnostic_notes": f"Errore pipeline: {e}",

                "api_calls_common": branch_calls_after.get("common", 0) - branch_calls_before.get("common", 0),
                "api_calls_vlm_guided": branch_calls_after.get("vlm_guided", 0) - branch_calls_before.get("vlm_guided", 0),
                "api_calls_ahp": branch_calls_after.get("ahp", 0) - branch_calls_before.get("ahp", 0),
                "api_calls_vlm_free": branch_calls_after.get("vlm_free", 0) - branch_calls_before.get("vlm_free", 0),
            }

            for risk_code in RISK_CUE_KEYS.keys():
                record[f"{risk_code}_raw"] = fallback_pack["raw"][risk_code]
                record[f"{risk_code}_transformed"] = fallback_pack["transformed"][risk_code]
                record[f"{risk_code}_weight"] = fallback_pack["weights"][risk_code]
                record[f"{risk_code}_contribution"] = fallback_pack["contributions"][risk_code]
                record[f"{risk_code}_contribution_pct"] = fallback_pack["contribution_pct"][risk_code]

        finally:
            if record is not None:
                records.append(record)

            if remote_file is not None:
                print(f"{EMOJI['delete']} Eliminazione file remoto Gemini...")
                delete_remote_clip(remote_file)

            cleanup_local_clip_with_retry(clip_path, retries=6, wait_seconds=1.0)

    print("=" * 80)
    print(f"{EMOJI['done']} Pipeline completata. Record prodotti: {len(records)}")

df = pd.DataFrame(records)
display(df.head(10))

## 14. Costruzione DataFrame risultati comparativi

Questa sezione costruisce il DataFrame finale con tutti i campi necessari per il confronto tra i due rami e per l'analisi dei contributi AHP.

In [ ]:

df = pd.DataFrame(records)
token_audit_df = pd.DataFrame(TOKEN_AUDIT_LOGS)
if len(df) > 0:
    print("Anteprima risultati ibridi:")
    display_cols = [
        "window_id", "start_time_s", "end_time_s",
        "instantaneous_risk_score_vlm", "dynamic_risk_score_vlm", "traffic_light_color_vlm",
        "instantaneous_risk_score_ahp", "dynamic_risk_score_ahp", "traffic_light_color_ahp",
        "instantaneous_risk_score_vlm_free", "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free",
    ]
    display(df[display_cols].head())
else:
    print("DataFrame vuoto. Eseguire prima la pipeline completa.")

if len(token_audit_df) > 0:
    print("Anteprima tracking token:")
    display(token_audit_df.head())


## 15. Analisi comparativa dei due rami

Questa sezione confronta score istantanei, score dinamici e semafori finali dei due rami. L'obiettivo è misurare convergenze e divergenze su base finestra-per-finestra.

In [ ]:
def same_traffic(a, b) -> str:
    if str(a) == "disabilitato" or str(b) == "disabilitato":
        return "n.d."
    return "accordo" if str(a) == str(b) else "disaccordo"


def branch_ordered_traffic_tuple(row: pd.Series) -> tuple:
    return (
        row.get("traffic_light_color_vlm_free", "disabilitato"),
        row.get("traffic_light_color_vlm", "disabilitato"),
        row.get("traffic_light_color_ahp", "disabilitato"),
    )


if len(df) > 0:
    branch_comparison_df = df[[
        "window_id",
        "start_time_s",
        "end_time_s",
        "instantaneous_risk_score_vlm_free",
        "instantaneous_risk_score_vlm",
        "instantaneous_risk_score_ahp",
        "dynamic_risk_score_vlm_free",
        "dynamic_risk_score_vlm",
        "dynamic_risk_score_ahp",
        "traffic_light_color_vlm_free",
        "traffic_light_color_vlm",
        "traffic_light_color_ahp",
    ]].copy()

    branch_comparison_df["diff_inst_free_vs_guided"] = (
        branch_comparison_df["instantaneous_risk_score_vlm_free"] - branch_comparison_df["instantaneous_risk_score_vlm"]
    ).abs().round(2)
    branch_comparison_df["diff_inst_free_vs_ahp"] = (
        branch_comparison_df["instantaneous_risk_score_vlm_free"] - branch_comparison_df["instantaneous_risk_score_ahp"]
    ).abs().round(2)
    branch_comparison_df["diff_inst_guided_vs_ahp"] = (
        branch_comparison_df["instantaneous_risk_score_vlm"] - branch_comparison_df["instantaneous_risk_score_ahp"]
    ).abs().round(2)

    branch_comparison_df["diff_dyn_free_vs_guided"] = (
        branch_comparison_df["dynamic_risk_score_vlm_free"] - branch_comparison_df["dynamic_risk_score_vlm"]
    ).abs().round(2)
    branch_comparison_df["diff_dyn_free_vs_ahp"] = (
        branch_comparison_df["dynamic_risk_score_vlm_free"] - branch_comparison_df["dynamic_risk_score_ahp"]
    ).abs().round(2)
    branch_comparison_df["diff_dyn_guided_vs_ahp"] = (
        branch_comparison_df["dynamic_risk_score_vlm"] - branch_comparison_df["dynamic_risk_score_ahp"]
    ).abs().round(2)

    branch_comparison_df["traffic_light_agreement_free_guided"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_vlm"]), axis=1
    )
    branch_comparison_df["traffic_light_agreement_free_ahp"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_ahp"]), axis=1
    )
    branch_comparison_df["traffic_light_agreement_guided_ahp"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm"], r["traffic_light_color_ahp"]), axis=1
    )
    branch_comparison_df["three_way_traffic_agreement"] = branch_comparison_df.apply(
        lambda r: "accordo_totale" if len({x for x in branch_ordered_traffic_tuple(r) if str(x) != "disabilitato"}) <= 1 else "disaccordo",
        axis=1,
    )

    branch_comparison_df["mean_dynamic_score_3_branches"] = branch_comparison_df[[
        "dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm", "dynamic_risk_score_ahp"
    ]].mean(axis=1, skipna=True).round(2)

    display(branch_comparison_df.head(10))
else:
    branch_comparison_df = pd.DataFrame()
    print("Nessun dato disponibile per il confronto finale tra i rami.")


## 16. Validazione di coerenza tra i due rami

Questa sezione conta quante volte i due rami convergono o divergono sul semaforo finale. È utile per verificare la coerenza delle due catene decisionali e per il debug sperimentale.

In [ ]:
def same_traffic(a, b) -> str:
    if str(a) == "disabilitato" or str(b) == "disabilitato":
        return "n.d."
    return "accordo" if str(a) == str(b) else "disaccordo"


def branch_ordered_traffic_tuple(row: pd.Series) -> tuple:
    return (
        row.get("traffic_light_color_vlm_free", "disabilitato"),
        row.get("traffic_light_color_vlm", "disabilitato"),
        row.get("traffic_light_color_ahp", "disabilitato"),
    )


if len(df) > 0:
    branch_comparison_df = df[[
        "window_id",
        "start_time_s",
        "end_time_s",
        "instantaneous_risk_score_vlm_free",
        "instantaneous_risk_score_vlm",
        "instantaneous_risk_score_ahp",
        "dynamic_risk_score_vlm_free",
        "dynamic_risk_score_vlm",
        "dynamic_risk_score_ahp",
        "traffic_light_color_vlm_free",
        "traffic_light_color_vlm",
        "traffic_light_color_ahp",
    ]].copy()

    branch_comparison_df["diff_inst_free_vs_guided"] = (
        branch_comparison_df["instantaneous_risk_score_vlm_free"] - branch_comparison_df["instantaneous_risk_score_vlm"]
    ).abs().round(2)
    branch_comparison_df["diff_inst_free_vs_ahp"] = (
        branch_comparison_df["instantaneous_risk_score_vlm_free"] - branch_comparison_df["instantaneous_risk_score_ahp"]
    ).abs().round(2)
    branch_comparison_df["diff_inst_guided_vs_ahp"] = (
        branch_comparison_df["instantaneous_risk_score_vlm"] - branch_comparison_df["instantaneous_risk_score_ahp"]
    ).abs().round(2)

    branch_comparison_df["diff_dyn_free_vs_guided"] = (
        branch_comparison_df["dynamic_risk_score_vlm_free"] - branch_comparison_df["dynamic_risk_score_vlm"]
    ).abs().round(2)
    branch_comparison_df["diff_dyn_free_vs_ahp"] = (
        branch_comparison_df["dynamic_risk_score_vlm_free"] - branch_comparison_df["dynamic_risk_score_ahp"]
    ).abs().round(2)
    branch_comparison_df["diff_dyn_guided_vs_ahp"] = (
        branch_comparison_df["dynamic_risk_score_vlm"] - branch_comparison_df["dynamic_risk_score_ahp"]
    ).abs().round(2)

    branch_comparison_df["traffic_light_agreement_free_guided"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_vlm"]), axis=1
    )
    branch_comparison_df["traffic_light_agreement_free_ahp"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_ahp"]), axis=1
    )
    branch_comparison_df["traffic_light_agreement_guided_ahp"] = branch_comparison_df.apply(
        lambda r: same_traffic(r["traffic_light_color_vlm"], r["traffic_light_color_ahp"]), axis=1
    )
    branch_comparison_df["three_way_traffic_agreement"] = branch_comparison_df.apply(
        lambda r: "accordo_totale" if len({x for x in branch_ordered_traffic_tuple(r) if str(x) != "disabilitato"}) <= 1 else "disaccordo",
        axis=1,
    )

    branch_comparison_df["mean_dynamic_score_3_branches"] = branch_comparison_df[[
        "dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm", "dynamic_risk_score_ahp"
    ]].mean(axis=1, skipna=True).round(2)

    display(branch_comparison_df.head(10))
else:
    branch_comparison_df = pd.DataFrame()
    print("Nessun dato disponibile per il confronto finale tra i rami.")


## 17. Analisi contributi AHP

Questa sezione costruisce una vista più leggibile dei contributi AHP per rischio, utile sia per l'interpretazione metodologica sia per l'export finale in Excel.

In [ ]:
def make_contribution_detail_df(df_: pd.DataFrame) -> pd.DataFrame:
    records_long = []
    for _, row in df_.iterrows():
        for risk_code in RISK_CUE_KEYS.keys():
            records_long.append({
                "window_id": row["window_id"],
                "start_time_s": row["start_time_s"],
                "end_time_s": row["end_time_s"],
                "risk_code": risk_code,
                "risk_label": RISK_LABELS[risk_code],
                "raw": row[f"{risk_code}_raw"],
                "transformed": row[f"{risk_code}_transformed"],
                "weight": row[f"{risk_code}_weight"],
                "contribution": row[f"{risk_code}_contribution"],
                "contribution_pct": row[f"{risk_code}_contribution_pct"],
            })
    return pd.DataFrame(records_long)


def summarize_mean_contributions(df_: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for risk_code in RISK_CUE_KEYS.keys():
        rows.append({
            "risk_code": risk_code,
            "risk_label": RISK_LABELS[risk_code],
            "mean_raw": round(df_[f"{risk_code}_raw"].mean(), 4),
            "mean_transformed": round(df_[f"{risk_code}_transformed"].mean(), 4),
            "weight": AHP_WEIGHTS[risk_code],
            "mean_contribution": round(df_[f"{risk_code}_contribution"].mean(), 4),
            "mean_contribution_pct": round(df_[f"{risk_code}_contribution_pct"].mean(), 2),
        })
    return pd.DataFrame(rows)


if len(df) > 0:
    contribution_cols = [f"{risk_code}_contribution" for risk_code in RISK_CUE_KEYS.keys()]
    contribution_pct_cols = [f"{risk_code}_contribution_pct" for risk_code in RISK_CUE_KEYS.keys()]

    risk_contribution_analysis_df = df[
        [
            "window_id",
            "start_time_s",
            "end_time_s",
            *contribution_cols,
            *contribution_pct_cols,
            "instantaneous_risk_score_ahp",
        ]
    ].copy()

    contribution_detail_df = make_contribution_detail_df(df)
    contribution_summary_df = summarize_mean_contributions(df)
    display(contribution_summary_df)
else:
    risk_contribution_analysis_df = pd.DataFrame()
    contribution_detail_df = pd.DataFrame()
    contribution_summary_df = pd.DataFrame()
    print("Nessun dato disponibile per l'analisi contributi AHP.")

## 18. Visualizzazioni semplici del confronto

Questa sezione fornisce una visualizzazione rapida dell'evoluzione temporale degli score dinamici dei tre rami e di alcune differenze tra coppie di rami.

In [ ]:
plot_df_all_branches = df.copy() if isinstance(globals().get("df", None), pd.DataFrame) else pd.DataFrame()

if plot_df_all_branches.empty:
    print("Nessun dato disponibile per il confronto dei tre rami della pipeline.")
elif "window_id" not in plot_df_all_branches.columns:
    print("Colonna 'window_id' non disponibile: impossibile costruire il confronto tra i tre rami.")
else:
    from matplotlib.ticker import MultipleLocator

    branch_specs_all = [
        ("dynamic_risk_score_vlm_free", "VLM libero", "#70AD47", "^"),
        ("dynamic_risk_score_vlm", "VLM guidato", "#4472C4", "o"),
        ("dynamic_risk_score_ahp", "VLM guidato + AHP", "#ED7D31", "s"),
    ]
    branch_specs_all = [spec for spec in branch_specs_all if spec[0] in plot_df_all_branches.columns]

    if len(branch_specs_all) == 0:
        print("Nessuna delle colonne dinamiche dei tre rami è disponibile per il grafico comparativo.")
    else:
        plot_df_all_branches = plot_df_all_branches.sort_values("window_id").reset_index(drop=True)
        plot_df_all_branches["window_id"] = pd.to_numeric(plot_df_all_branches["window_id"], errors="coerce")

        x_positions = list(range(len(plot_df_all_branches)))

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        for col_name, label, color, marker in branch_specs_all:
            series = pd.to_numeric(plot_df_all_branches[col_name], errors="coerce")
            if series.notna().any():
                axes[0].plot(
                    x_positions,
                    series,
                    marker=marker,
                    linewidth=2,
                    markersize=6,
                    label=label,
                    color=color,
                )

        axes[0].set_title("Confronto score dinamici nel tempo")
        axes[0].set_xlabel("Numero finestra temporale")
        axes[0].set_ylabel("Score dinamico")
        axes[0].set_ylim(0, 100)

        if x_positions:
            axes[0].set_xticks(x_positions)
            axes[1].set_xticks(x_positions)
            axes[0].set_xlim(min(x_positions) - 0.5, max(x_positions) + 0.5)
            axes[1].set_xlim(min(x_positions) - 0.5, max(x_positions) + 0.5)

        axes[0].tick_params(axis="x", labelbottom=True)

        axes[0].yaxis.set_major_locator(MultipleLocator(20))
        axes[0].yaxis.set_minor_locator(MultipleLocator(10))
        axes[1].yaxis.set_minor_locator(MultipleLocator(5))

        axes[0].grid(True, which="major", axis="both", alpha=0.30)
        axes[0].grid(True, which="minor", axis="both", alpha=0.15)

        if axes[0].lines:
            axes[0].legend()
        else:
            axes[0].text(
                0.5,
                0.5,
                "Nessuna serie disponibile da visualizzare.",
                ha="center",
                va="center",
                transform=axes[0].transAxes,
            )

        diff_specs = [
            (
                "dynamic_risk_score_vlm_free",
                "dynamic_risk_score_vlm",
                "free_vs_guided",
                "#E92727",
            ),
            (
                "dynamic_risk_score_vlm_free",
                "dynamic_risk_score_ahp",
                "free_vs_ahp",
                "#FFE100",
            ),
            (
                "dynamic_risk_score_vlm",
                "dynamic_risk_score_ahp",
                "guided_vs_ahp",
                "#03F6C9",
            ),
        ]

        available_diff_specs = [
            spec for spec in diff_specs
            if spec[0] in plot_df_all_branches.columns and spec[1] in plot_df_all_branches.columns
        ]

        if available_diff_specs:
            n_pairs = len(available_diff_specs)
            bar_width = 0.8 / max(n_pairs, 1)

            for idx, (col_a, col_b, label, color) in enumerate(available_diff_specs):
                diff_series = (
                    pd.to_numeric(plot_df_all_branches[col_a], errors="coerce")
                    - pd.to_numeric(plot_df_all_branches[col_b], errors="coerce")
                ).abs().round(2)

                x_offset = [x - 0.4 + (idx + 0.5) * bar_width for x in x_positions]

                axes[1].bar(
                    x_offset,
                    diff_series.fillna(0.0),
                    width=bar_width,
                    color=color,
                    alpha=0.9,
                    label=label,
                )

            axes[1].legend()
        else:
            axes[1].text(
                0.5,
                0.5,
                "Differenze non disponibili: mancano le colonne necessarie.",
                ha="center",
                va="center",
                transform=axes[1].transAxes,
            )

        axes[1].set_title("Differenze assolute tra coppie di rami")
        axes[1].set_xlabel("Numero finestra temporale")
        axes[1].set_ylabel("Differenza assoluta score dinamico")
        axes[1].grid(True, which="major", axis="both", alpha=0.30)
        axes[1].grid(True, which="minor", axis="both", alpha=0.15)

        plt.tight_layout()
        plt.show()

### Confronto dedicato tra VLM libero e VLM guidato

Questa sottosezione aggiunge una visualizzazione separata focalizzata solo sui due rami VLM, mantenendo invariata la visualizzazione comparativa a tre rami.


In [ ]:
plot_df_vlm_only = df.copy() if isinstance(globals().get("df", None), pd.DataFrame) else pd.DataFrame()

if plot_df_vlm_only.empty:
    print("Nessun dato disponibile per il confronto dedicato tra VLM libero e VLM guidato.")
elif "window_id" not in plot_df_vlm_only.columns:
    print("Colonna 'window_id' non disponibile: impossibile costruire il confronto dedicato tra i due rami VLM.")
else:
    from matplotlib.ticker import MultipleLocator

    branch_specs_vlm_only = [
        ("dynamic_risk_score_vlm_free", "VLM libero", "#70AD47", "^"),
        ("dynamic_risk_score_vlm", "VLM guidato", "#4472C4", "o"),
    ]
    branch_specs_vlm_only = [spec for spec in branch_specs_vlm_only if spec[0] in plot_df_vlm_only.columns]

    if len(branch_specs_vlm_only) == 0:
        print("Nessuna delle colonne dinamiche dei due rami VLM è disponibile per il grafico dedicato.")
    else:
        plot_df_vlm_only = plot_df_vlm_only.sort_values("window_id").reset_index(drop=True)
        plot_df_vlm_only["window_id"] = pd.to_numeric(plot_df_vlm_only["window_id"], errors="coerce")

        x_positions = list(range(len(plot_df_vlm_only)))

        fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        for col_name, label, color, marker in branch_specs_vlm_only:
            series = pd.to_numeric(plot_df_vlm_only[col_name], errors="coerce")
            if series.notna().any():
                axes[0].plot(
                    x_positions,
                    series,
                    marker=marker,
                    linewidth=2,
                    markersize=6,
                    label=label,
                    color=color,
                )

        axes[0].set_title("Confronto score dinamici nel tempo - solo rami VLM")
        axes[0].set_xlabel("Numero finestra temporale")
        axes[0].set_ylabel("Score dinamico")
        axes[0].set_ylim(0, 100)

        if x_positions:
            axes[0].set_xticks(x_positions)
            axes[1].set_xticks(x_positions)
            axes[0].set_xlim(min(x_positions) - 0.5, max(x_positions) + 0.5)
            axes[1].set_xlim(min(x_positions) - 0.5, max(x_positions) + 0.5)

        axes[0].tick_params(axis="x", labelbottom=True)

        axes[0].yaxis.set_major_locator(MultipleLocator(20))
        axes[0].yaxis.set_minor_locator(MultipleLocator(10))
        axes[1].yaxis.set_minor_locator(MultipleLocator(5))

        axes[0].grid(True, which="major", axis="both", alpha=0.30)
        axes[0].grid(True, which="minor", axis="both", alpha=0.15)

        if axes[0].lines:
            axes[0].legend()
        else:
            axes[0].text(
                0.5,
                0.5,
                "Nessuna serie disponibile da visualizzare.",
                ha="center",
                va="center",
                transform=axes[0].transAxes,
            )

        if all(col in plot_df_vlm_only.columns for col in ["dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm"]):
            diff_series_vlm_only = (
                pd.to_numeric(plot_df_vlm_only["dynamic_risk_score_vlm_free"], errors="coerce")
                - pd.to_numeric(plot_df_vlm_only["dynamic_risk_score_vlm"], errors="coerce")
            ).abs().round(2)

            if diff_series_vlm_only.notna().any():
                axes[1].bar(
                    x_positions,
                    diff_series_vlm_only.fillna(0.0),
                    color="#E92727",
                    alpha=0.9,
                    width=0.8,
                    label="|VLM libero - VLM guidato|",
                )
                axes[1].legend()
            else:
                axes[1].text(
                    0.5,
                    0.5,
                    "Differenza non disponibile.",
                    ha="center",
                    va="center",
                    transform=axes[1].transAxes,
                )
        else:
            missing_cols_vlm_only = [c for c in ["dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm"] if c not in plot_df_vlm_only.columns]
            axes[1].text(
                0.5,
                0.5,
                "Differenza non disponibile: mancano le colonne " + ", ".join(missing_cols_vlm_only),
                ha="center",
                va="center",
                transform=axes[1].transAxes,
            )

        axes[1].set_title("Differenza assoluta tra i due rami VLM")
        axes[1].set_xlabel("Numero finestra temporale")
        axes[1].set_ylabel("Differenza assoluta score dinamico")
        axes[1].grid(True, which="major", axis="both", alpha=0.30)
        axes[1].grid(True, which="minor", axis="both", alpha=0.15)

        plt.tight_layout()
        plt.show()

## 19. Esportazione Excel con dashboard comparativa e file token

L'Excel finale principale continua a raccogliere i risultati dei rami della pipeline, mantenendo lo stile di esportazione originario. Il ramo **VLM libero** viene ora incluso in modo esplicito nel confronto finale a **tre rami** per score istantanei, score dinamici, semafori e indicatori di accordo/disaccordo.

In aggiunta, resta esportato un secondo file Excel separato dedicato al monitoraggio dei token per tutti i blocchi VLM eseguiti nei diversi rami.


In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.chart import LineChart, Reference
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook



def same_traffic(a, b) -> str:
    if str(a) == "disabilitato" or str(b) == "disabilitato":
        return "n.d."
    return "accordo" if str(a) == str(b) else "disaccordo"



def branch_ordered_traffic_tuple(row: pd.Series) -> tuple:
    return (
        row.get("traffic_light_color_vlm_free", "disabilitato"),
        row.get("traffic_light_color_vlm", "disabilitato"),
        row.get("traffic_light_color_ahp", "disabilitato"),
    )



def safe_subset(df_source: pd.DataFrame, columns: list) -> pd.DataFrame:
    existing = [c for c in columns if c in df_source.columns]
    if not existing:
        return pd.DataFrame()
    return df_source[existing].copy()



def compact_text(value, max_len=90):
    """
    Nessun troncamento: manteniamo il contenuto completo nella cella.
    """
    if value is None:
        return None
    return str(value).replace("\r\n", "\n").replace("\r", "\n")



def compact_dataframe_for_excel(df_source: pd.DataFrame, text_limit=90) -> pd.DataFrame:
    """
    Compatibilità col codice esistente, ma senza compattazione/troncamento.
    """
    df_out = df_source.copy()
    for col in df_out.columns:
        if df_out[col].dtype == "object":
            df_out[col] = df_out[col].apply(lambda x: compact_text(x, max_len=text_limit))
    return df_out



def drop_api_call_columns(df_source: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = [
        "branch_api_call_count",
        "api_calls_common",
        "api_calls_vlm_guided",
        "api_calls_ahp",
        "api_calls_vlm_free",
    ]
    existing = [c for c in cols_to_remove if c in df_source.columns]
    if existing:
        return df_source.drop(columns=existing)
    return df_source.copy()



def first_existing_column(df_source: pd.DataFrame, candidates: list) -> Optional[str]:
    for c in candidates:
        if c in df_source.columns:
            return c
    return None



def rename_if_present(df_source: pd.DataFrame, rename_map: dict) -> pd.DataFrame:
    effective_map = {k: v for k, v in rename_map.items() if k in df_source.columns}
    if not effective_map:
        return df_source.copy()
    return df_source.rename(columns=effective_map).copy()



def traffic_fill(color_name: str):
    color_name = str(color_name).strip().lower()
    mapping = {
        "verde": "C6EFCE",
        "giallo": "FFF2CC",
        "arancione": "FCE4D6",
        "rosso": "F4CCCC",
        "disabilitato": "D9D9D9",
        "n.d.": "EDEDED",
    }
    return PatternFill(fill_type="solid", fgColor=mapping.get(color_name, "FFFFFF"))



def is_textual_excel_value(value) -> bool:
    return isinstance(value, str) and value is not None



def is_long_text_column(col_name: str) -> bool:
    if col_name is None:
        return False

    col_name = str(col_name)

    explicit_long_cols = {
        "free_descriptive_causal_output_json",
        "vlm_free_scene_json",
        "free_scene_json",
        "vlm_free_scoring_json",
        "free_score_json",
        "score_rationale_vlm_free",
        "descriptive_prompt_output_json",
        "common_descriptive_json",
        "scene_json",
        "question_set_prompt_output_json",
        "common_question_set_json",
        "diagnostic_json",
        "causal_prompt_output_json",
        "common_causal_json",
        "causal_json",
        "vlm_guided_scoring_json",
        "vlm_score_json",
        "score_rationale_vlm",
        "diagnostic_notes",
        "error",
    }

    if col_name in explicit_long_cols:
        return True

    long_markers = [
        "json",
        "rationale",
        "notes",
        "description",
        "descriptive",
        "diagnostic",
        "causal",
        "scene",
        "error",
    ]
    col_lower = col_name.lower()
    return any(marker in col_lower for marker in long_markers)



def style_ws_compact(ws):
    """
    Versione MOLTO compatta:
    - contenuto completo salvato nella cella
    - celle visivamente più piccole
    - righe quasi fisse
    - colonne JSON/rationale più strette
    """
    header_fill = PatternFill(fill_type="solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True, size=8)
    body_font = Font(size=7)
    thin = Side(style="thin", color="D9D9D9")

    ws.freeze_panes = "A2"
    ws.sheet_view.zoomScale = 80
    ws.sheet_view.showGridLines = True

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True, shrink_to_fit=False)
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = body_font
            cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True, shrink_to_fit=False)
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    narrow_cols = {
        "window_id": 6,
        "start_time_s": 7,
        "end_time_s": 7,
        "window_duration_s": 7,
        "instantaneous_risk_score_vlm_free": 8,
        "instantaneous_risk_score_vlm": 8,
        "instantaneous_risk_score_ahp": 8,
        "dynamic_risk_score_vlm_free": 8,
        "dynamic_risk_score_vlm": 8,
        "dynamic_risk_score_ahp": 8,
        "traffic_light_color_vlm_free": 7,
        "traffic_light_color_vlm": 7,
        "traffic_light_color_ahp": 7,
        "diff_inst_free_vs_guided": 8,
        "diff_inst_free_vs_ahp": 8,
        "diff_inst_guided_vs_ahp": 8,
        "diff_dyn_free_vs_guided": 8,
        "diff_dyn_free_vs_ahp": 8,
        "diff_dyn_guided_vs_ahp": 8,
        "traffic_light_agreement_free_guided": 10,
        "traffic_light_agreement_free_ahp": 10,
        "traffic_light_agreement_guided_ahp": 10,
        "three_way_traffic_agreement": 11,
        "mean_dynamic_score_3_branches": 9,
        "execution_order": 7,
        "timestamp": 14,
        "branch_name": 10,
        "block_name": 12,
        "model_name": 11,
        "input_tokens": 8,
        "output_tokens": 8,
        "total_tokens": 8,
        "usage_source": 10,
        "status": 8,
        "branch_api_call_count": 9,
        "cue_confidence": 8,
        "S1_raw": 7,
        "S1_transformed": 8,
        "S1_weight": 7,
        "S1_contribution": 8,
        "S1_contribution_pct": 8,
        "S2_raw": 7,
        "S2_transformed": 8,
        "S2_weight": 7,
        "S2_contribution": 8,
        "S2_contribution_pct": 8,
        "S3_raw": 7,
        "S3_transformed": 8,
        "S3_weight": 7,
        "S3_contribution": 8,
        "S3_contribution_pct": 8,
        "S4_raw": 7,
        "S4_transformed": 8,
        "S4_weight": 7,
        "S4_contribution": 8,
        "S4_contribution_pct": 8,
        "S5_raw": 7,
        "S5_transformed": 8,
        "S5_weight": 7,
        "S5_contribution": 8,
        "S5_contribution_pct": 8,
        "S6_raw": 7,
        "S6_transformed": 8,
        "S6_weight": 7,
        "S6_contribution": 8,
        "S6_contribution_pct": 8,
        "S7_raw": 7,
        "S7_transformed": 8,
        "S7_weight": 7,
        "S7_contribution": 8,
        "S7_contribution_pct": 8,
    }

    medium_cols = {
        "score_rationale_vlm_free": 16,
        "score_rationale_vlm": 16,
        "diagnostic_notes": 18,
        "error": 16,
        "vlm_free_scoring_json": 14,
        "free_score_json": 14,
        "vlm_guided_scoring_json": 14,
        "vlm_score_json": 14,
    }

    long_cols = {
        "free_descriptive_causal_output_json": 18,
        "vlm_free_scene_json": 18,
        "free_scene_json": 18,
        "descriptive_prompt_output_json": 18,
        "common_descriptive_json": 18,
        "scene_json": 18,
        "question_set_prompt_output_json": 18,
        "common_question_set_json": 18,
        "diagnostic_json": 18,
        "causal_prompt_output_json": 18,
        "common_causal_json": 18,
        "causal_json": 18,
    }

    headers = [c.value for c in ws[1]]

    for idx, col_name in enumerate(headers, start=1):
        if col_name in narrow_cols:
            width = narrow_cols[col_name]
        elif col_name in medium_cols:
            width = medium_cols[col_name]
        elif col_name in long_cols:
            width = long_cols[col_name]
        elif is_long_text_column(col_name):
            width = 18
        else:
            max_len = 0
            for row in range(1, min(ws.max_row, 12) + 1):
                value = ws.cell(row=row, column=idx).value
                value_len = len(str(value)) if value is not None else 0
                if value_len > max_len:
                    max_len = value_len
            width = min(max(max_len + 1, 6), 12)

        ws.column_dimensions[get_column_letter(idx)].width = width

    ws.row_dimensions[1].height = 16

    for r in range(2, ws.max_row + 1):
        ws.row_dimensions[r].height = 34



def color_columns_by_groups(ws, groups: dict):
    group_fills = {
        "time": PatternFill(fill_type="solid", fgColor="D9EAF7"),
        "free": PatternFill(fill_type="solid", fgColor="E2F0D9"),
        "guided": PatternFill(fill_type="solid", fgColor="D9E1F2"),
        "ahp": PatternFill(fill_type="solid", fgColor="FCE4D6"),
        "comparison": PatternFill(fill_type="solid", fgColor="FFF2CC"),
        "agreement": PatternFill(fill_type="solid", fgColor="EADCF8"),
        "summary": PatternFill(fill_type="solid", fgColor="EDEDED"),
    }

    header_names = [cell.value for cell in ws[1]]
    col_map = {name: idx + 1 for idx, name in enumerate(header_names)}

    for group_name, columns in groups.items():
        fill = group_fills.get(group_name)
        if fill is None:
            continue
        for col_name in columns:
            if col_name in col_map:
                col_idx = col_map[col_name]
                ws.cell(row=1, column=col_idx).fill = fill
                ws.cell(row=1, column=col_idx).font = Font(color="000000", bold=True, size=8)

    traffic_cols = [
        "traffic_light_color_vlm_free",
        "traffic_light_color_vlm",
        "traffic_light_color_ahp",
    ]
    agreement_cols = [
        "traffic_light_agreement_free_guided",
        "traffic_light_agreement_free_ahp",
        "traffic_light_agreement_guided_ahp",
        "three_way_traffic_agreement",
        "traffic_light_agreement_vlm_free_vs_vlm_guided",
    ]

    for col_name in traffic_cols:
        if col_name in col_map:
            col_idx = col_map[col_name]
            for row in range(2, ws.max_row + 1):
                val = ws.cell(row=row, column=col_idx).value
                ws.cell(row=row, column=col_idx).fill = traffic_fill(val)

    for col_name in agreement_cols:
        if col_name in col_map:
            col_idx = col_map[col_name]
            for row in range(2, ws.max_row + 1):
                val = str(ws.cell(row=row, column=col_idx).value).strip().lower()
                if val == "accordo":
                    ws.cell(row=row, column=col_idx).fill = PatternFill(fill_type="solid", fgColor="C6EFCE")
                elif val == "disaccordo":
                    ws.cell(row=row, column=col_idx).fill = PatternFill(fill_type="solid", fgColor="F4CCCC")
                elif val == "accordo_totale":
                    ws.cell(row=row, column=col_idx).fill = PatternFill(fill_type="solid", fgColor="A9D18E")
                else:
                    ws.cell(row=row, column=col_idx).fill = PatternFill(fill_type="solid", fgColor="EDEDED")



def add_line_chart(ws, title, y_title, data_col_idxs, category_col_idx=1, anchor="N2", colors=None):
    if ws.max_row < 2:
        return

    chart = LineChart()
    chart.title = title
    chart.y_axis.title = y_title
    chart.x_axis.title = "Window ID"
    chart.height = 7
    chart.width = 14
    chart.style = 2

    cats = Reference(ws, min_col=category_col_idx, min_row=2, max_row=ws.max_row)
    for i, col_idx in enumerate(data_col_idxs):
        data = Reference(ws, min_col=col_idx, min_row=1, max_row=ws.max_row)
        chart.add_data(data, titles_from_data=True)
        if colors and i < len(colors):
            try:
                chart.series[i].graphicalProperties.line.solidFill = colors[i]
                chart.series[i].graphicalProperties.line.width = 18000
            except Exception:
                pass

    chart.set_categories(cats)
    ws.add_chart(chart, anchor)



def ensure_vlm_only_raw_columns(base_df: pd.DataFrame) -> pd.DataFrame:
    result = base_df.copy() if isinstance(base_df, pd.DataFrame) else pd.DataFrame()
    raw_cols = [f"S{i}_raw" for i in range(1, 8)]

    for col in raw_cols:
        if col not in result.columns:
            result[col] = np.nan

    diagnostic_candidates = [
        "diagnostic_json",
        "common_question_set_json",
        "question_set_prompt_output_json",
        "guided_diagnostic_json",
    ]

    def _coerce_to_dict(value):
        if isinstance(value, dict):
            return value
        if isinstance(value, str) and value.strip():
            try:
                parsed = json.loads(value)
                return parsed if isinstance(parsed, dict) else {}
            except Exception:
                return {}
        return {}

    for idx in result.index:
        if all(pd.notna(result.at[idx, col]) for col in raw_cols):
            continue

        diagnostic_json = {}
        for candidate in diagnostic_candidates:
            if candidate in result.columns:
                diagnostic_json = _coerce_to_dict(result.at[idx, candidate])
                if diagnostic_json:
                    break

        extracted_raw = extract_raw_risk_values(diagnostic_json) if diagnostic_json else {}
        for i in range(1, 8):
            col = f"S{i}_raw"
            if pd.isna(result.at[idx, col]):
                result.at[idx, col] = extracted_raw.get(f"S{i}", np.nan)

    return result



def build_vlm_only_export_dataframe(base_df: pd.DataFrame) -> pd.DataFrame:
    raw_cols = [f"S{i}_raw" for i in range(1, 8)]
    base_columns = [
        "window_id", "start_time_s", "end_time_s", "window_duration_s",
        "free_scene_json", "free_score_json",
        "instantaneous_risk_score_vlm_free", "score_rationale_vlm_free",
        "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free",
        "scene_json", "diagnostic_json", "causal_json", "vlm_score_json",
        "instantaneous_risk_score_vlm", "score_rationale_vlm",
        "dynamic_risk_score_vlm", "traffic_light_color_vlm",
        "diff_inst_vlm_free_vs_vlm_guided", "diff_dyn_vlm_free_vs_vlm_guided",
        "traffic_light_agreement_vlm_free_vs_vlm_guided",
    ] + raw_cols

    if not isinstance(base_df, pd.DataFrame) or base_df.empty:
        return pd.DataFrame(columns=base_columns)

    work_df = ensure_vlm_only_raw_columns(base_df)
    existing_cols = [c for c in base_columns if c in work_df.columns and "ahp" not in c.lower()]
    export_df = work_df[existing_cols].copy()

    if all(c in work_df.columns for c in ["instantaneous_risk_score_vlm_free", "instantaneous_risk_score_vlm"]):
        export_df["diff_inst_vlm_free_vs_vlm_guided"] = (
            pd.to_numeric(work_df["instantaneous_risk_score_vlm_free"], errors="coerce")
            - pd.to_numeric(work_df["instantaneous_risk_score_vlm"], errors="coerce")
        ).abs().round(2)

    if all(c in work_df.columns for c in ["dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm"]):
        export_df["diff_dyn_vlm_free_vs_vlm_guided"] = (
            pd.to_numeric(work_df["dynamic_risk_score_vlm_free"], errors="coerce")
            - pd.to_numeric(work_df["dynamic_risk_score_vlm"], errors="coerce")
        ).abs().round(2)

    if all(c in work_df.columns for c in ["traffic_light_color_vlm_free", "traffic_light_color_vlm"]):
        export_df["traffic_light_agreement_vlm_free_vs_vlm_guided"] = work_df.apply(
            lambda r: same_traffic(
                r.get("traffic_light_color_vlm_free", "disabilitato"),
                r.get("traffic_light_color_vlm", "disabilitato"),
            ),
            axis=1,
        )

    preferred_order = [c for c in base_columns if c in export_df.columns]
    remaining_cols = [c for c in export_df.columns if c not in preferred_order and "ahp" not in c.lower()]
    export_df = export_df[[c for c in preferred_order + remaining_cols if "ahp" not in c.lower()]]
    return export_df



if len(df) > 0:
    df_results_only = drop_api_call_columns(df)

    branch_comparison_df = safe_subset(df_results_only, [
        "window_id",
        "start_time_s",
        "end_time_s",
        "instantaneous_risk_score_vlm_free",
        "instantaneous_risk_score_vlm",
        "instantaneous_risk_score_ahp",
        "dynamic_risk_score_vlm_free",
        "dynamic_risk_score_vlm",
        "dynamic_risk_score_ahp",
        "traffic_light_color_vlm_free",
        "traffic_light_color_vlm",
        "traffic_light_color_ahp",
    ]).copy()

    if all(c in branch_comparison_df.columns for c in [
        "instantaneous_risk_score_vlm_free", "instantaneous_risk_score_vlm"
    ]):
        branch_comparison_df["diff_inst_free_vs_guided"] = (
            branch_comparison_df["instantaneous_risk_score_vlm_free"]
            - branch_comparison_df["instantaneous_risk_score_vlm"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "instantaneous_risk_score_vlm_free", "instantaneous_risk_score_ahp"
    ]):
        branch_comparison_df["diff_inst_free_vs_ahp"] = (
            branch_comparison_df["instantaneous_risk_score_vlm_free"]
            - branch_comparison_df["instantaneous_risk_score_ahp"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "instantaneous_risk_score_vlm", "instantaneous_risk_score_ahp"
    ]):
        branch_comparison_df["diff_inst_guided_vs_ahp"] = (
            branch_comparison_df["instantaneous_risk_score_vlm"]
            - branch_comparison_df["instantaneous_risk_score_ahp"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "dynamic_risk_score_vlm_free", "dynamic_risk_score_vlm"
    ]):
        branch_comparison_df["diff_dyn_free_vs_guided"] = (
            branch_comparison_df["dynamic_risk_score_vlm_free"]
            - branch_comparison_df["dynamic_risk_score_vlm"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "dynamic_risk_score_vlm_free", "dynamic_risk_score_ahp"
    ]):
        branch_comparison_df["diff_dyn_free_vs_ahp"] = (
            branch_comparison_df["dynamic_risk_score_vlm_free"]
            - branch_comparison_df["dynamic_risk_score_ahp"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "dynamic_risk_score_vlm", "dynamic_risk_score_ahp"
    ]):
        branch_comparison_df["diff_dyn_guided_vs_ahp"] = (
            branch_comparison_df["dynamic_risk_score_vlm"]
            - branch_comparison_df["dynamic_risk_score_ahp"]
        ).abs().round(2)

    if all(c in branch_comparison_df.columns for c in [
        "traffic_light_color_vlm_free", "traffic_light_color_vlm"
    ]):
        branch_comparison_df["traffic_light_agreement_free_guided"] = branch_comparison_df.apply(
            lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_vlm"]),
            axis=1,
        )

    if all(c in branch_comparison_df.columns for c in [
        "traffic_light_color_vlm_free", "traffic_light_color_ahp"
    ]):
        branch_comparison_df["traffic_light_agreement_free_ahp"] = branch_comparison_df.apply(
            lambda r: same_traffic(r["traffic_light_color_vlm_free"], r["traffic_light_color_ahp"]),
            axis=1,
        )

    if all(c in branch_comparison_df.columns for c in [
        "traffic_light_color_vlm", "traffic_light_color_ahp"
    ]):
        branch_comparison_df["traffic_light_agreement_guided_ahp"] = branch_comparison_df.apply(
            lambda r: same_traffic(r["traffic_light_color_vlm"], r["traffic_light_color_ahp"]),
            axis=1,
        )

    branch_comparison_df["three_way_traffic_agreement"] = branch_comparison_df.apply(
        lambda r: "accordo_totale" if len({x for x in branch_ordered_traffic_tuple(r) if str(x) != "disabilitato"}) <= 1 else "disaccordo",
        axis=1,
    )

    dynamic_cols_for_mean = [
        c for c in [
            "dynamic_risk_score_vlm_free",
            "dynamic_risk_score_vlm",
            "dynamic_risk_score_ahp",
        ] if c in branch_comparison_df.columns
    ]
    if dynamic_cols_for_mean:
        branch_comparison_df["mean_dynamic_score_3_branches"] = (
            branch_comparison_df[dynamic_cols_for_mean].mean(axis=1, skipna=True).round(2)
        )

    if "token_audit_df" not in globals() or token_audit_df is None or len(token_audit_df) == 0:
        token_audit_df = pd.DataFrame(TOKEN_AUDIT_LOGS)

    if "contribution_detail_df" not in globals():
        contribution_detail_df = pd.DataFrame()

    if "contribution_summary_df" not in globals():
        contribution_summary_df = pd.DataFrame()

    if "risk_contribution_analysis_df" not in globals():
        risk_contribution_analysis_df = pd.DataFrame()

    results_excel_path = OUT_DIR / "pipeline_results_complete.xlsx"
    tokens_excel_path = OUT_DIR / "pipeline_token_audit.xlsx"
    vlm_only_excel_path = OUT_DIR / "pipeline_results_vlm_only.xlsx"

    main_result_scoring_json_cols = [
        c for c in ["vlm_free_scoring_json", "free_score_json", "vlm_guided_scoring_json", "vlm_score_json"]
        if c in df_results_only.columns
    ]
    all_results_main_df = df_results_only.drop(columns=main_result_scoring_json_cols, errors="ignore").copy()

    scoring_json_debug_cols = [c for c in main_result_scoring_json_cols if c in df_results_only.columns]
    scoring_json_debug_base_cols = [c for c in ["window_id", "start_time_s", "end_time_s"] if c in df_results_only.columns]
    scoring_json_debug_df = safe_subset(df_results_only, scoring_json_debug_base_cols + scoring_json_debug_cols)

    all_results_export_df = compact_dataframe_for_excel(all_results_main_df, text_limit=90)
    branch_comparison_export_df = compact_dataframe_for_excel(branch_comparison_df, text_limit=60)

    free_scene_col = first_existing_column(df_results_only, [
        "free_descriptive_causal_output_json",
        "vlm_free_scene_json",
        "free_scene_json",
    ])
    free_score_col = first_existing_column(df_results_only, [
        "vlm_free_scoring_json",
        "free_score_json",
    ])

    guided_scene_col = first_existing_column(df_results_only, [
        "descriptive_prompt_output_json",
        "common_descriptive_json",
        "scene_json",
    ])
    guided_diagnostic_col = first_existing_column(df_results_only, [
        "question_set_prompt_output_json",
        "common_question_set_json",
        "diagnostic_json",
    ])
    guided_causal_col = first_existing_column(df_results_only, [
        "causal_prompt_output_json",
        "common_causal_json",
        "causal_json",
    ])
    guided_score_col = first_existing_column(df_results_only, [
        "vlm_guided_scoring_json",
        "vlm_score_json",
    ])

    vlm_free_branch_cols = [
        "window_id",
        "start_time_s",
        "end_time_s",
        free_scene_col,
        "instantaneous_risk_score_vlm_free",
        "score_rationale_vlm_free",
        "dynamic_risk_score_vlm_free",
        "traffic_light_color_vlm_free",
    ]
    vlm_free_branch_cols = [c for c in vlm_free_branch_cols if c is not None]

    vlm_guided_branch_cols = [
        "window_id",
        "start_time_s",
        "end_time_s",
        guided_scene_col,
        guided_diagnostic_col,
        guided_causal_col,
        "instantaneous_risk_score_vlm",
        "score_rationale_vlm",
        "dynamic_risk_score_vlm",
        "traffic_light_color_vlm",
    ]
    vlm_guided_branch_cols = [c for c in vlm_guided_branch_cols if c is not None]

    ahp_branch_cols = [
        "window_id",
        "start_time_s",
        "end_time_s",
        guided_scene_col,
        guided_diagnostic_col,
        guided_causal_col,
        "instantaneous_risk_score_ahp",
        "dynamic_risk_score_ahp",
        "traffic_light_color_ahp",
    ]
    ahp_branch_cols = [c for c in ahp_branch_cols if c is not None]

    vlm_free_branch_df = compact_dataframe_for_excel(
        safe_subset(df_results_only, vlm_free_branch_cols),
        text_limit=80
    )
    vlm_guided_branch_df = compact_dataframe_for_excel(
        safe_subset(df_results_only, vlm_guided_branch_cols),
        text_limit=80
    )
    ahp_branch_df = compact_dataframe_for_excel(
        safe_subset(df_results_only, ahp_branch_cols),
        text_limit=80
    )

    if free_scene_col and free_scene_col != "free_scene_json":
        vlm_free_branch_df = rename_if_present(vlm_free_branch_df, {free_scene_col: "free_scene_json"})
    if free_score_col and free_score_col in vlm_free_branch_df.columns and free_score_col != "free_score_json":
        vlm_free_branch_df = rename_if_present(vlm_free_branch_df, {free_score_col: "free_score_json"})

    guided_rename_map = {}
    if guided_scene_col and guided_scene_col != "scene_json":
        guided_rename_map[guided_scene_col] = "scene_json"
    if guided_diagnostic_col and guided_diagnostic_col != "diagnostic_json":
        guided_rename_map[guided_diagnostic_col] = "diagnostic_json"
    if guided_causal_col and guided_causal_col != "causal_json":
        guided_rename_map[guided_causal_col] = "causal_json"
    if guided_score_col and guided_score_col in vlm_guided_branch_df.columns and guided_score_col != "vlm_score_json":
        guided_rename_map[guided_score_col] = "vlm_score_json"

    vlm_guided_branch_df = rename_if_present(vlm_guided_branch_df, guided_rename_map)

    ahp_rename_map = {}
    if guided_scene_col and guided_scene_col in ahp_branch_df.columns and guided_scene_col != "scene_json":
        ahp_rename_map[guided_scene_col] = "scene_json"
    if guided_diagnostic_col and guided_diagnostic_col in ahp_branch_df.columns and guided_diagnostic_col != "diagnostic_json":
        ahp_rename_map[guided_diagnostic_col] = "diagnostic_json"
    if guided_causal_col and guided_causal_col in ahp_branch_df.columns and guided_causal_col != "causal_json":
        ahp_rename_map[guided_causal_col] = "causal_json"
    ahp_branch_df = rename_if_present(ahp_branch_df, ahp_rename_map)

    token_audit_export_df = compact_dataframe_for_excel(token_audit_df, text_limit=80)
    contribution_detail_export_df = compact_dataframe_for_excel(contribution_detail_df, text_limit=80)
    contribution_summary_export_df = compact_dataframe_for_excel(contribution_summary_df, text_limit=80)
    risk_contribution_analysis_export_df = compact_dataframe_for_excel(risk_contribution_analysis_df, text_limit=80)

    vlm_only_export_df = build_vlm_only_export_dataframe(df_results_only)
    vlm_only_export_df_compact = compact_dataframe_for_excel(vlm_only_export_df, text_limit=80)

    with pd.ExcelWriter(results_excel_path, engine="openpyxl") as writer:
        all_results_export_df.to_excel(writer, sheet_name="all_results", index=False)
        branch_comparison_export_df.to_excel(writer, sheet_name="branch_comparison", index=False)
        vlm_free_branch_df.to_excel(writer, sheet_name="vlm_free_branch", index=False)
        vlm_guided_branch_df.to_excel(writer, sheet_name="vlm_guided_branch", index=False)
        ahp_branch_df.to_excel(writer, sheet_name="ahp_branch", index=False)

        if len(scoring_json_debug_df) > 0:
            compact_dataframe_for_excel(scoring_json_debug_df, text_limit=90).to_excel(
                writer, sheet_name="scoring_json_debug", index=False
            )

        if len(token_audit_export_df) > 0:
            token_audit_export_df.to_excel(writer, sheet_name="token_audit", index=False)
        else:
            pd.DataFrame([{"message": "Nessun dato token disponibile"}]).to_excel(
                writer, sheet_name="token_audit", index=False
            )

        if len(risk_contribution_analysis_export_df) > 0:
            risk_contribution_analysis_export_df.to_excel(writer, sheet_name="ahp_risk_analysis", index=False)

        if len(contribution_detail_export_df) > 0:
            contribution_detail_export_df.to_excel(writer, sheet_name="ahp_contrib_detail", index=False)

        if len(contribution_summary_export_df) > 0:
            contribution_summary_export_df.to_excel(writer, sheet_name="ahp_contrib_summary", index=False)

    wb = load_workbook(results_excel_path)

    ws_all = wb["all_results"]
    style_ws_compact(ws_all)
    color_columns_by_groups(ws_all, {
        "time": ["window_id", "start_time_s", "end_time_s", "window_duration_s"],
        "free": [
            "free_descriptive_causal_output_json", "vlm_free_scene_json", "free_scene_json",
            "vlm_free_scoring_json", "free_score_json",
            "instantaneous_risk_score_vlm_free", "score_rationale_vlm_free",
            "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free"
        ],
        "guided": [
            "descriptive_prompt_output_json", "common_descriptive_json", "scene_json",
            "question_set_prompt_output_json", "common_question_set_json", "diagnostic_json",
            "causal_prompt_output_json", "common_causal_json", "causal_json",
            "vlm_guided_scoring_json", "vlm_score_json",
            "instantaneous_risk_score_vlm", "score_rationale_vlm",
            "dynamic_risk_score_vlm", "traffic_light_color_vlm"
        ],
        "ahp": [
            "instantaneous_risk_score_ahp", "dynamic_risk_score_ahp", "traffic_light_color_ahp"
        ],
    })

    all_headers = [c.value for c in ws_all[1]]
    idx_map_all = {name: i + 1 for i, name in enumerate(all_headers)}
    dyn_cols_all = [
        idx_map_all[c] for c in [
            "dynamic_risk_score_vlm_free",
            "dynamic_risk_score_vlm",
            "dynamic_risk_score_ahp"
        ] if c in idx_map_all
    ]
    if "window_id" in idx_map_all and len(dyn_cols_all) > 0:
        add_line_chart(
            ws_all,
            title="Dynamic Risk Score - Tre rami",
            y_title="Dynamic risk score",
            data_col_idxs=dyn_cols_all,
            category_col_idx=idx_map_all["window_id"],
            anchor="P2",
            colors=["70AD47", "4472C4", "ED7D31"],
        )

    ws_comp = wb["branch_comparison"]
    style_ws_compact(ws_comp)
    color_columns_by_groups(ws_comp, {
        "time": ["window_id", "start_time_s", "end_time_s"],
        "free": ["instantaneous_risk_score_vlm_free", "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free"],
        "guided": ["instantaneous_risk_score_vlm", "dynamic_risk_score_vlm", "traffic_light_color_vlm"],
        "ahp": ["instantaneous_risk_score_ahp", "dynamic_risk_score_ahp", "traffic_light_color_ahp"],
        "comparison": [
            "diff_inst_free_vs_guided", "diff_inst_free_vs_ahp", "diff_inst_guided_vs_ahp",
            "diff_dyn_free_vs_guided", "diff_dyn_free_vs_ahp", "diff_dyn_guided_vs_ahp"
        ],
        "agreement": [
            "traffic_light_agreement_free_guided", "traffic_light_agreement_free_ahp",
            "traffic_light_agreement_guided_ahp", "three_way_traffic_agreement"
        ],
        "summary": ["mean_dynamic_score_3_branches"],
    })

    comp_headers = [c.value for c in ws_comp[1]]
    idx_map_comp = {name: i + 1 for i, name in enumerate(comp_headers)}
    dyn_cols_comp = [
        idx_map_comp[c] for c in [
            "dynamic_risk_score_vlm_free",
            "dynamic_risk_score_vlm",
            "dynamic_risk_score_ahp"
        ] if c in idx_map_comp
    ]
    if "window_id" in idx_map_comp and len(dyn_cols_comp) > 0:
        add_line_chart(
            ws_comp,
            title="Confronto punteggi dinamici - Tre rami",
            y_title="Dynamic risk score",
            data_col_idxs=dyn_cols_comp,
            category_col_idx=idx_map_comp["window_id"],
            anchor="Z2",
            colors=["70AD47", "4472C4", "ED7D31"],
        )

    branch_sheet_specs = [
        ("vlm_free_branch", "dynamic_risk_score_vlm_free", "Ramo VLM libero - Dynamic risk score", "70AD47"),
        ("vlm_guided_branch", "dynamic_risk_score_vlm", "Ramo VLM guidato - Dynamic risk score", "4472C4"),
        ("ahp_branch", "dynamic_risk_score_ahp", "Ramo AHP - Dynamic risk score", "ED7D31"),
    ]

    for sheet_name, dyn_col_name, chart_title, color_hex in branch_sheet_specs:
        if sheet_name in wb.sheetnames:
            ws_branch = wb[sheet_name]
            style_ws_compact(ws_branch)

            if sheet_name == "vlm_free_branch":
                color_columns_by_groups(ws_branch, {
                    "time": ["window_id", "start_time_s", "end_time_s"],
                    "free": [
                        "free_scene_json",
                        "instantaneous_risk_score_vlm_free", "score_rationale_vlm_free",
                        "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free"
                    ],
                })
            elif sheet_name == "vlm_guided_branch":
                color_columns_by_groups(ws_branch, {
                    "time": ["window_id", "start_time_s", "end_time_s"],
                    "guided": [
                        "scene_json", "diagnostic_json", "causal_json",
                        "instantaneous_risk_score_vlm", "score_rationale_vlm",
                        "dynamic_risk_score_vlm", "traffic_light_color_vlm"
                    ],
                })
            elif sheet_name == "ahp_branch":
                color_columns_by_groups(ws_branch, {
                    "time": ["window_id", "start_time_s", "end_time_s"],
                    "guided": ["scene_json", "diagnostic_json", "causal_json"],
                    "ahp": [
                        "instantaneous_risk_score_ahp", "dynamic_risk_score_ahp", "traffic_light_color_ahp"
                    ],
                })

            branch_headers = [c.value for c in ws_branch[1]]
            idx_map_branch = {name: i + 1 for i, name in enumerate(branch_headers)}

            if "window_id" in idx_map_branch and dyn_col_name in idx_map_branch:
                add_line_chart(
                    ws_branch,
                    title=chart_title,
                    y_title="Dynamic risk score",
                    data_col_idxs=[idx_map_branch[dyn_col_name]],
                    category_col_idx=idx_map_branch["window_id"],
                    anchor="L2",
                    colors=[color_hex],
                )

    for sheet_name in ["token_audit", "ahp_risk_analysis", "ahp_contrib_detail", "ahp_contrib_summary", "scoring_json_debug"]:
        if sheet_name in wb.sheetnames:
            style_ws_compact(wb[sheet_name])

    if "token_audit" in wb.sheetnames:
        color_columns_by_groups(wb["token_audit"], {
            "time": ["execution_order", "timestamp", "window_id"],
            "summary": [
                "branch_name", "block_name", "model_name",
                "input_tokens", "output_tokens", "total_tokens",
                "usage_source", "status", "error", "branch_api_call_count"
            ],
        })

    wb.save(results_excel_path)

    with pd.ExcelWriter(tokens_excel_path, engine="openpyxl") as writer:
        if len(token_audit_export_df) > 0:
            token_audit_export_df.to_excel(writer, sheet_name="token_audit", index=False)
        else:
            pd.DataFrame([{"message": "Nessun dato token disponibile"}]).to_excel(
                writer, sheet_name="token_audit", index=False
            )

    wb_tokens = load_workbook(tokens_excel_path)
    ws_tokens = wb_tokens["token_audit"]
    style_ws_compact(ws_tokens)
    color_columns_by_groups(ws_tokens, {
        "time": ["execution_order", "timestamp", "window_id"],
        "summary": [
            "branch_name", "block_name", "model_name",
            "input_tokens", "output_tokens", "total_tokens",
            "usage_source", "status", "error", "branch_api_call_count"
        ],
    })
    wb_tokens.save(tokens_excel_path)

    with pd.ExcelWriter(vlm_only_excel_path, engine="openpyxl") as writer:
        vlm_only_export_df_compact.to_excel(writer, sheet_name="vlm_only_results", index=False)

    wb_vlm_only = load_workbook(vlm_only_excel_path)
    ws_vlm_only = wb_vlm_only["vlm_only_results"]
    style_ws_compact(ws_vlm_only)
    color_columns_by_groups(ws_vlm_only, {
        "time": ["window_id", "start_time_s", "end_time_s", "window_duration_s"],
        "free": [
            "free_scene_json", "free_score_json",
            "instantaneous_risk_score_vlm_free", "score_rationale_vlm_free",
            "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free"
        ],
        "guided": [
            "scene_json", "diagnostic_json", "causal_json", "vlm_score_json",
            "instantaneous_risk_score_vlm", "score_rationale_vlm",
            "dynamic_risk_score_vlm", "traffic_light_color_vlm"
        ],
        "comparison": [
            "diff_inst_vlm_free_vs_vlm_guided",
            "diff_dyn_vlm_free_vs_vlm_guided"
        ],
        "agreement": [
            "traffic_light_agreement_vlm_free_vs_vlm_guided"
        ],
        "summary": [f"S{i}_raw" for i in range(1, 8)],
    })

    vlm_only_headers = [c.value for c in ws_vlm_only[1]]
    idx_map_vlm_only = {name: i + 1 for i, name in enumerate(vlm_only_headers)}
    dyn_cols_vlm_only = [
        idx_map_vlm_only[c] for c in [
            "dynamic_risk_score_vlm_free",
            "dynamic_risk_score_vlm",
        ] if c in idx_map_vlm_only
    ]
    if "window_id" in idx_map_vlm_only and len(dyn_cols_vlm_only) > 0:
        add_line_chart(
            ws_vlm_only,
            title="Dynamic Risk Score - Solo rami VLM",
            y_title="Dynamic risk score",
            data_col_idxs=dyn_cols_vlm_only,
            category_col_idx=idx_map_vlm_only["window_id"],
            anchor="X2",
            colors=["70AD47", "4472C4"],
        )

    wb_vlm_only.save(vlm_only_excel_path)

    print(f"File Excel risultati salvato in: {results_excel_path}")
    print(f"File Excel token salvato in: {tokens_excel_path}")
    print(f"File Excel solo VLM salvato in: {vlm_only_excel_path}")

else:
    branch_comparison_df = pd.DataFrame()
    print("Nessun dato disponibile per il confronto finale tra i rami.")

## 20. Note finali

Il notebook resta una versione estesa e conservativa di `vlm_risk_pipeline_hybrid`. In particolare:
- la parte iniziale comune resta centrata su ingestione video e windowing temporale;
- i rami esistenti vengono mantenuti;
- il nuovo ramo **VLM libero** si inserisce dopo la base comune, con logica confrontabile;
- ogni ramo può essere attivato o disattivato tramite flag;
- il tracking dei token è stato aggiunto senza alterare la filosofia generale di retry, parsing difensivo ed export finale.


In [ ]:
# Verifica finale conservativa dei campi richiesti
if "df" in globals() and isinstance(df, pd.DataFrame):
    required_cols = [
        "instantaneous_risk_score_vlm_free", "score_rationale_vlm_free", "dynamic_risk_score_vlm_free", "traffic_light_color_vlm_free",
        "instantaneous_risk_score_vlm", "score_rationale_vlm", "dynamic_risk_score_vlm", "traffic_light_color_vlm",
    ]
    print("Colonne richieste presenti:", [c for c in required_cols if c in df.columns])
    missing = [c for c in required_cols if c not in df.columns]
    print("Colonne richieste mancanti:", missing)
    show_cols = [c for c in required_cols if c in df.columns]
    if show_cols:
        display(df[show_cols].head())
